# КИВО: анализ

**Это основной блокнот анализа.** Здесь проверяются гипотезы и исследовательские задачи из `hypotheses.md`. Данные уже очищены в блокноте `cleaning.ipynb`; здесь они только загружаются.

**Перед запуском** в папке должны лежать `data_clean.pkl` и `data_imputed.pkl`. Если их нет или изменились правила очистки, сначала полностью выполните `cleaning.ipynb`.

**Содержание**
1. Библиотеки и настройки
2. Очищенные данные
3. Переменные для анализа
4. Флаги качества: какие значения брать
5. Аналитические выборки
6. Восстановленные наборы и объединение результатов
7. Как проверяется каждая гипотеза
8. ИВ1. Структура и социальные различия занятости
9. ИВ2. Занятость и обучение
10. ИВ3. Польза и трудности работы
11. ИВ4. Опыт стажировок
12. ИВ5. Занятость и здоровье
13. ИВ6. Планы распределения времени и трудоустройства
14. ИВ7. Изменения между волнами
15. Сводка решений по гипотезам

План анализа (данные, методы и правила решения для каждой гипотезы) — `analysis_plan.md`.

## 1. Библиотеки и настройки

Все библиотеки для анализа подключаются здесь.

In [1]:
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import patsy
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display
from scipy import stats
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 80)
sns.set_theme(style='whitegrid', context='notebook')

SEED = 2026
rng = np.random.default_rng(SEED)

## 2. Очищенные данные

- `data_clean.pkl` — все записи выгрузки с исходными переменными, статусами ответов (`*_st`), аналитическими переменными и флагами качества (`q_*`, `qsrc_*`). Для анализа берём только рабочий массив (`in_study`).
- `data_imputed.pkl` — 7 наборов с восстановленными пропусками (раздел 6).

In [2]:
full = pd.read_pickle('data_clean.pkl')
d = full[full['in_study']].copy()
IMPUTED = pd.read_pickle('data_imputed.pkl')

print(f'Рабочий массив: {len(d)} записей (из {len(full)} в выгрузке)')
pd.DataFrame({'записей': [d['in_w3'].sum(), d['in_w4'].sum(), d['panel'].sum()]},
             index=['3-я волна', '4-я волна', 'обе волны'])

Рабочий массив: 2022 записей (из 5831 в выгрузке)


,записей
3-я волна,1224
4-я волна,1513
обе волны,715


**Вывод.** В анализе 2 022 записи: 1 224 участника 3-й волны, 1 513 — 4-й, 715 — обеих волн. Строки без ответов уже исключены.

## 3. Переменные для анализа

Окончание `_3` / `_4` — номер волны. Все аналитические переменные построены только из содержательных ответов: «затрудняюсь», отказ, неприменимые и пустые ответы в них — пропуски (`NaN`), а их тип хранится в статусе `*_st` исходного вопроса.

In [3]:
CODEBOOK = {
    # занятость
    'emp': 'есть оплачиваемая работа (1/0)',
    'fmt': 'формат занятости: нет работы / полный день / неполный день / подработки',
    'hours': 'рабочих часов за прошлую неделю',
    'motive_raw': 'мотив работы: исходные варианты анкеты',
    'motive_g14': 'мотив работы, группировка для Г1.4',
    'motive_g35': 'мотив работы, группировка для Г3.4–3.5 (только 4-я волна)',
    'motive_rec': 'мотив работы с перенесёнными открытыми ответами (проверка устойчивости, 4-я волна)',
    'pay_tuition': 'мотив «оплата обучения» (1/0)',
    'sphere': 'сфера работы',
    'sphere_rec': 'сфера с перенесёнными открытыми ответами (4-я волна)',
    'fit': 'соответствие работы специальности, 1–4 (4-я волна)',
    'benefit': 'работа помогает усваивать учебный материал, 1–4 (4-я волна)',
    'harm': 'из-за работы страдает успеваемость, 1–4 (4-я волна)',
    'difficulty': 'трудно совмещать работу и учёбу, 1–4 (4-я волна)',
    'benefit_difficulty': 'сочетание пользы и трудностей, З3.6 (4-я волна)',
    'intent': 'намерен искать работу (1/0), только неработающие',
    # обучение
    'mark': 'категория оценок за последнюю сессию, 1–4',
    'sat': 'удовлетворённость обучением, 1–4',
    'edu3': 'повторный выбор образования',
    'drop_any': 'задумывался об отчислении (1/0)',
    'reason1': 'причина: разочарование в вузе (1/0)',
    'reason2': 'причина: разочарование в направлении (1/0)',
    'reason3': 'причина: академические трудности (1/0)',
    'reason4': 'причина: отношения с одногруппниками (1/0)',
    'reason5': 'причина: отношения с преподавателями (1/0)',
    'reason6': 'причина: экономические трудности семьи (1/0)',
    'reason7': 'причина: трудности совмещения с работой (1/0)',
    'reason_other': 'причина: «другое» (1/0, 4-я волна)',
    'reasons_cmp': 'отмечена хотя бы одна из семи причин — сопоставимое между волнами основание',
    # стажировки
    'intern': 'был опыт стажировки (1/0)',
    'intern_course': 'курс прохождения стажировки',
    'intern_paid': 'стажировка оплачивалась (1/0)',
    'intern_motive_raw': 'мотив стажировки (3-я волна)',
    'intern_motive_g44': 'мотив стажировки, группировка для Г4.4 (3-я волна)',
    'intern_fit': 'соответствие стажировки направлению, 1–4',
    'intern_benefit': 'польза стажировки для учёбы, 1–4 (3-я волна)',
    'intern_harm': 'ущерб успеваемости из-за стажировки, 1–4 (3-я волна)',
    'intern_difficulty': 'трудности совмещения со стажировкой, 1–4 (3-я волна)',
    'work_intern': 'сочетание текущей работы и опыта стажировки, З4.6',
    # здоровье
    'sleep_dur_wd': 'ночной интервал сна в будни, часы',
    'sleep_dur_we': 'ночной интервал сна в выходные, часы',
    'sjl': 'сдвиг середины сна между буднями и выходными, часы',
    'pe': 'занимается спортом (1/0)',
    'pe_freq': 'частота занятий спортом, 1–5, больше = чаще (3-я волна)',
    'smoke': 'курит (1/0)',
    'alco': 'употребляет алкоголь (1/0)',
    # планы
    'alloc_учёба': 'планируемая доля времени: учёба (0–1, 3-я волна)',
    'alloc_работа': 'планируемая доля времени: оплачиваемая работа (0–1, 3-я волна)',
    'alloc_клубы': 'планируемая доля времени: студенческие клубы (0–1, 3-я волна)',
    'alloc_волонтёрство': 'планируемая доля времени: волонтёрство (0–1, 3-я волна)',
    'alloc_курсы': 'планируемая доля времени: внешние курсы (0–1, 3-я волна)',
    'alloc_остальное': 'планируемая доля времени: остальное (0–1, 3-я волна)',
    'plan_work_pos': 'планирует время на работу, доля > 0 (1/0, 3-я волна)',
}
BETWEEN_WAVES = {
    'trans4': 'переход занятости между волнами',
    'start_work': 'начал работать к 4-й волне (1/0) — среди не работавших в 3-й',
    'start_fmt': 'формат работы начавших работать',
    'd_hours': 'изменение рабочих часов (4-я − 3-я)',
    'd_sleep_wd': 'изменение сна в будни (4-я − 3-я)',
    'new_drop_4': 'впервые задумался об отчислении (среди не думавших в 3-й)',
    'smoke_change': 'курение: 3-я → 4-я волна',
    'alco_change': 'алкоголь: 3-я → 4-я волна',
    'pe_change': 'спорт: 3-я → 4-я волна',
}
CONTROL_VARS = {
    'sex_c': 'пол', 'major_c': 'направление (референтная — инженерные; коды 4 и 8 исключены)',
    'funding_c': 'финансирование: бюджет / платное / целевое',
    'ses': 'материальное положение семьи, 1–6', 'ses4': 'материальное положение, 4 категории',
}
SERVICE_VARS = {
    'in_w3': 'есть ответы 3-й волны', 'in_w4': 'есть ответы 4-й волны', 'panel': 'есть ответы обеих волн',
    'partial_4w': 'частично заполненная анкета 4-й волны', 'tech_A': 'профиль A', 'tech_B': 'профиль B',
    'dup_flag': 'полностью совпадающая строка', 'wave_flag_conflict': 'признак волны расходится с ответами',
}


def describe_var(var, description, group):
    x = d[var]
    values = (f'{x.min():g} – {x.max():g}' if pd.api.types.is_numeric_dtype(x) and x.dtype != bool
              else ', '.join(map(str, x.dropna().unique()[:5])))
    return {'группа': group, 'переменная': var, 'описание': description, 'заполнено': int(x.notna().sum()),
            'значения': values}


rows = []
for stem, description in CODEBOOK.items():
    for w in ['3', '4']:
        if f'{stem}_{w}' in d:
            rows.append(describe_var(f'{stem}_{w}', description, 'по волнам'))
rows += [describe_var(v, t, 'между волнами') for v, t in BETWEEN_WAVES.items()]
rows += [describe_var(v, t, 'социальные характеристики') for v, t in CONTROL_VARS.items()]
rows += [describe_var(v, t, 'служебные признаки') for v, t in SERVICE_VARS.items()]
codebook = pd.DataFrame(rows).set_index(['группа', 'переменная'])
codebook

описание  \
группа                    переменная                                                                                              
по волнам                 emp_3                                                                  есть оплачиваемая работа (1/0)   
                          emp_4                                                                  есть оплачиваемая работа (1/0)   
                          fmt_3                         формат занятости: нет работы / полный день / неполный день / подработки   
                          fmt_4                         формат занятости: нет работы / полный день / неполный день / подработки   
                          hours_3                                                               рабочих часов за прошлую неделю   
                          hours_4                                                               рабочих часов за прошлую неделю   
                          motive_raw_3                                                   мотив работы: исходные варианты анкеты   
                          motive_raw_4                                                   мотив работы: исходные варианты анкеты   
                          motive_g14_3                                                       мотив работы, группировка для Г1.4   
                          motive_g14_4                                                       мотив работы, группировка для Г1.4   
                          motive_g35_3                                мотив работы, группировка для Г3.4–3.5 (только 4-я волна)   
                          motive_g35_4                                мотив работы, группировка для Г3.4–3.5 (только 4-я волна)   
                          motive_rec_4          мотив работы с перенесёнными открытыми ответами (проверка устойчивости, 4-я ...   
                          pay_tuition_3                                                           мотив «оплата обучения» (1/0)   
                          pay_tuition_4                                                           мотив «оплата обучения» (1/0)   
                          sphere_3                                                                                 сфера работы   
                          sphere_4                                                                                 сфера работы   
                          sphere_rec_4                                     сфера с перенесёнными открытыми ответами (4-я волна)   
                          fit_4                                              соответствие работы специальности, 1–4 (4-я волна)   
                          benefit_4                                 работа помогает усваивать учебный материал, 1–4 (4-я волна)   
                          harm_4                                            из-за работы страдает успеваемость, 1–4 (4-я волна)   
                          difficulty_4                                         трудно совмещать работу и учёбу, 1–4 (4-я волна)   
                          benefit_difficulty_4                                  сочетание пользы и трудностей, З3.6 (4-я волна)   
                          intent_3                                             намерен искать работу (1/0), только неработающие   
                          intent_4                                             намерен искать работу (1/0), только неработающие   
                          mark_3                                                      категория оценок за последнюю сессию, 1–4   
                          mark_4                                                      категория оценок за последнюю сессию, 1–4   
                          sat_3                                                                удовлетворённость обучением, 1–4   
                          sat_4                                                                удовлетворённость обучением, 1–4   
                          edu3_3                          

**Вывод.** Таблица — справочник: какие переменные есть, что они значат и сколько у каждой заполненных значений в рабочем массиве.

## 4. Флаги качества: какие значения брать

У каждой измеряемой переменной есть флаг `q_<имя>` (код качества) и `qsrc_<имя>` (причины).

| Код | Значение | Основной анализ | Проверка устойчивости |
|---|---|---|---|
| 0 | исходное надёжное | да | да |
| 1 | исправленное (перенесённый открытый ответ) | да | да |
| 2 | восстановленное | да | да |
| 3–5 | сомнительное (часы-выбросы, цензурированный сон, частичная анкета и т. п.) | да | **нет** |
| 6 | ненадёжное (невозможное значение) | нет | нет |
| 9 | пропуск | нет | нет |
| 99 | не применимо | нет | нет |

Правило: основной анализ — на кодах 0–5; затем тот же анализ повторяется без сомнительных значений (коды 0–2), и результаты сравниваются.

In [4]:
QUALITY_ALIAS = {'funding_c': 'funding', 'ses': 'ses1', 'ses4': 'ses1', 'sex_c': 'sex', 'major_c': 'bigmajor'}
PREFIX_ALIAS = [('sleep_dur_wd', 'sleep_wd'), ('sleep_dur_we', 'sleep_we'), ('reason', 'reasons'),
                ('intern_motive', 'intern_motive'), ('motive', 'motive'), ('pay_tuition', 'motive'),
                ('alloc', 'alloc'), ('plan_work', 'alloc'), ('sphere', 'sphere')]


def quality_name(var):
    '''Имя переменной, флаг которой относится к var (None, если флага нет).'''
    if f'q_{var}' in d:
        return var
    if var in QUALITY_ALIAS:
        return QUALITY_ALIAS[var]
    for prefix, target in PREFIX_ALIAS:
        if var.startswith(prefix) and f'q_{target}_{var[-1]}' in d:
            return f'{target}_{var[-1]}'
    return None


def usable(var, robust=False):
    '''Строки, где значение var можно использовать: коды 0–5 (основной анализ) или 0–2 (устойчивость).'''
    name = quality_name(var)
    ok = d[var].notna()
    if name is not None:
        ok &= d[f'q_{name}'].isin([0, 1, 2] if robust else [0, 1, 2, 3, 4, 5])
    return ok


# Пример: рабочие часы 4-й волны
pd.DataFrame({'основной анализ': [usable('hours_4').sum()], 'без сомнительных': [usable('hours_4', robust=True).sum()]},
             index=['hours_4'])

,основной анализ,без сомнительных
hours_4,427,347


**Вывод.** Для часов 4-й волны в основной анализ входят 427 значений, без сомнительных — 347: остальные — нулевые часы, выбросы или частичные анкеты.

## 5. Аналитические выборки

Для каждой проверки задаются условие выборки, необходимые переменные (независимая и зависимая) и дополнительные поправки из гипотезы. Определения скопированы из `cleaning.ipynb`, раздел 14.

Функция `sample()` возвращает таблицу для проверки: только строки с нужными значениями и с учётом флагов.

In [5]:
in3, in4 = d['in_w3'], d['in_w4']
work3, work4 = in3 & d['emp_3'].eq(1), in4 & d['emp_4'].eq(1)
CONTROLS = ['sex_c', 'major_c', 'funding_c', 'ses']
IMPUTABLE = {'sex_c', 'major_c', 'funding_c', 'ses'}
HOURS_IMPUTABLE = {'hours_3': d['work_hours_st'].isin(['MISS', 'INVALID']),
                   'hours_4': d['work_hours_4w_st'].isin(['MISS', 'INVALID'])}
intern3 = in3 & d['intern_3'].eq(1)
not_working_3 = d['start_work'].notna()


def full_day_or_none(w):
    return d[f'fmt_{w}'].isin(['полный день', 'нет работы'])


# проверка: (условие выборки, необходимые переменные, дополнительные поправки)
SAMPLES = {}
for w, work, in_wave in [('3', work3, in3), ('4', work4, in4)]:
    h = f'hours_{w}'
    SAMPLES.update({
        f'З1.1, {w} в.': (in_wave, [f'fmt_{w}'], []),
        f'Г1.2, {w} в.': (in_wave & ~d['funding_c'].eq('целевое'), [f'emp_{w}', 'funding_c'], []),
        f'Г1.3, {w} в.': (in_wave, [f'emp_{w}', 'ses'], []),
        f'Г1.4, {w} в.': (work, [f'motive_g14_{w}', 'ses'], []),
        f'З1.5, {w} в.': (work, [f'motive_raw_{w}', 'funding_c'], []),
        f'З1.6, {w} в.': (work, [f'sphere_{w}'], []),
        f'Г2.1, {w} в.': (work, [h, f'mark_{w}'], []),
        f'Г2.2, {w} в.': (work, [h, f'sat_{w}'], []),
        f'Г2.3, {w} в.': (in_wave & full_day_or_none(w), [f'fmt_{w}', f'drop_any_{w}'], []),
        f'Г2.4, {w} в.': (work & d[f'drop_any_{w}'].eq(1), [h, f'reason7_{w}'], []),
        f'З2.5, {w} в.': (in_wave, [f'fmt_{w}', f'edu3_{w}'], []),
        f'З2.6, {w} в.': (in_wave & d[f'drop_any_{w}'].eq(1) & d[f'reasons_cmp_{w}'], [f'fmt_{w}', f'reason1_{w}'], []),
        f'З3.7, {w} в., оценки': (work, [f'motive_raw_{w}', f'mark_{w}'], [h]),
        f'З3.7, {w} в., удовл.': (work, [f'motive_raw_{w}', f'sat_{w}'], [h]),
        f'З4.1, {w} в.': (in_wave, [f'intern_{w}'], []),
        f'З4.6, {w} в., оценки': (in_wave, [f'work_intern_{w}', f'mark_{w}'], []),
        f'Г5.1, {w} в.': (work, [h, f'sleep_dur_wd_{w}'], []),
        f'Г5.2, {w} в.': (work, [h, f'sjl_{w}'], []),
        f'Г5.3, {w} в.': (in_wave & full_day_or_none(w), [f'fmt_{w}', f'pe_{w}'], []),
        f'З5.5, {w} в., курение': (in_wave, [f'fmt_{w}', f'smoke_{w}'], []),
        f'З5.5, {w} в., алкоголь': (in_wave, [f'fmt_{w}', f'alco_{w}'], []),
        f'З5.6, {w} в.': (in_wave, [f'fmt_{w}', f'sleep_dur_wd_{w}', f'sjl_{w}'], []),
    })
SAMPLES.update({
    'З2.6, 4 в., с «другим»': (in4 & d['drop_any_4'].eq(1) & d['reasons_4w_any_st'].eq('VALID'),
                               ['fmt_4', 'reason_other_4'], []),
    'Г3.1': (work4, ['fit_4', 'benefit_4'], ['hours_4', 'motive_g35_4']),
    'Г3.2': (work4, ['fit_4', 'harm_4'], ['hours_4', 'motive_g35_4']),
    'Г3.3': (work4, ['hours_4', 'difficulty_4'], ['fit_4', 'motive_g35_4']),
    'Г3.4': (work4, ['motive_g35_4', 'benefit_4'], ['hours_4', 'fit_4']),
    'Г3.5': (work4, ['motive_g35_4', 'difficulty_4'], ['hours_4', 'fit_4']),
    'З3.6': (work4, ['benefit_difficulty_4'], []),
    'З3.8': (work4, ['harm_4', 'mark_4'], ['hours_4']),
    'Г4.2': (intern3, ['intern_fit_3', 'intern_benefit_3'], ['intern_paid_3', 'intern_motive_g44_3']),
    'Г4.3': (intern3, ['intern_fit_3', 'intern_harm_3'], ['intern_paid_3', 'intern_motive_g44_3']),
    'Г4.4': (intern3, ['intern_motive_g44_3', 'intern_benefit_3'], ['intern_fit_3', 'intern_paid_3']),
    'З4.5, 3 в.': (intern3, ['intern_paid_3'], []),
    'З4.5, 4 в.': (in4 & d['intern_4'].eq(1), ['intern_paid_4', 'intern_fit_4'], []),
    'Г5.4': (work3 & d['pe_3'].eq(1), ['hours_3', 'pe_freq_3'], []),
    'З6.1': (in3, ['alloc_работа_3'], []),
    'Г6.2': (not_working_3, ['start_work', 'intent_3'], []),
    'Г6.3': (not_working_3, ['start_work', 'plan_work_pos_3'], []),
    'З6.4, оценки': (d['panel'], ['alloc_работа_3', 'mark_3', 'mark_4', 'emp_3'], []),
    'З6.4, удовл.': (d['panel'], ['alloc_работа_3', 'sat_3', 'sat_4', 'emp_3'], []),
    'З7.1': (d['trans4'].notna(), ['trans4'], []),
    'Г7.2': (not_working_3, ['start_work', 'sat_3', 'sat_4'], []),
    'Г7.3': (not_working_3, ['start_work', 'mark_3', 'mark_4'], []),
    'Г7.4': (d['trans4'].eq('работал в обеих'), ['hours_3', 'hours_4', 'sleep_dur_wd_3', 'sleep_dur_wd_4'], []),
    'Г7.5': (not_working_3, ['start_work', 'intern_3'], []),
    'З7.6, курение': (d['trans4'].notna(), ['smoke_change'], []),
    'З7.6, отчисление впервые': (d['trans4'].notna(), ['new_drop_4'], []),
})



def sample(name, controls=True, robust=False):
    '''Данные для проверки name: условие выборки, нужные переменные без пропусков, флаги качества.'''
    mask, required, extra = SAMPLES[name]
    columns = list(dict.fromkeys(required + extra + (CONTROLS if controls else [])))
    rows = mask.copy()
    for var in columns:
        rows &= usable(var, robust)
    return d.loc[rows, columns]


sizes = pd.DataFrame({name: {'переменные гипотезы': len(sample(name, controls=False)),
                             '+ социальные поправки': len(sample(name)),
                             '+ поправки, без сомнительных': len(sample(name, robust=True))}
                      for name in SAMPLES}).T
sizes

,переменные гипотезы,+ социальные поправки,"+ поправки, без сомнительных"
"З1.1, 3 в.",1211,776,774
"Г1.2, 3 в.",897,742,742
"Г1.3, 3 в.",793,776,776
"Г1.4, 3 в.",342,337,337
"З1.5, 3 в.",410,346,345
"З1.6, 3 в.",429,282,281
"Г2.1, 3 в.",444,281,268
"Г2.2, 3 в.",432,273,260
"Г2.3, 3 в.",727,461,460
"Г2.4, 3 в.",203,132,125


**Вывод.** Объёмы выборок для всех проверок. Поправка на материальное положение сокращает выборки сильнее всего, потому что `ses1` пропущен у трети участников; восстановленные наборы (раздел 6) возвращают эти записи.

## 6. Восстановленные наборы и объединение результатов

В `data_imputed.pkl` — 7 наборов по семействам гипотез. В каждом m вариантов данных с разными правдоподобными значениями пропусков.

| Набор | Для каких проверок |
|---|---|
| `w3`, `w4` | все участники волны (ИВ1, ИВ2, ИВ5) |
| `work3`, `work4` | работающие в волне (часы, мотивы, польза и трудности) |
| `intern3` | проходившие стажировку, 3-я волна (ИВ4) |
| `nw3` | не работавшие в 3-й волне (Г6.2, Г6.3, Г7.2, Г7.3, Г7.5) |
| `workboth` | работавшие в обеих волнах (Г7.4) |

Восстановлены только социальные характеристики и рабочие часы со статусом «пропуск» или «невозможное значение». Зависимые переменные не восстанавливаются.

Порядок работы: (1) `imputed_datasets()` даёт m таблиц; (2) модель оценивается на каждой; (3) `fit_pooled()` объединяет оценки по правилам Рубина.

In [6]:
SEX = {1: 'мужской', 2: 'женский'}
FUNDING = {1: 'бюджет', 2: 'платное', 3: 'целевое'}
MAJOR = {1: 'естественные', 2: 'инженерные', 5: 'общественные', 7: 'гуманитарные'}


def imputed_datasets(set_name):
    '''m восстановленных таблиц: переменные из d, социальные характеристики и часы — восстановленные.'''
    long = IMPUTED[set_name]
    datasets = []
    for _, imp in long[long['.imp'] > 0].groupby('.imp'):
        imp = imp.set_index('row')
        frame = d.loc[imp.index].copy()
        frame['sex_c'] = pd.Categorical(imp['sex'].map(SEX), categories=d['sex_c'].cat.categories)
        frame['funding_c'] = pd.Categorical(imp['funding'].map(FUNDING), categories=d['funding_c'].cat.categories)
        frame['major_c'] = pd.Categorical(imp['bigmajor'].map(MAJOR), categories=d['major_c'].cat.categories)
        frame['ses'] = imp['ses']
        for w in ['3', '4']:
            if f'loghours_{w}' in imp:
                frame[f'hours_{w}'] = np.expm1(imp[f'loghours_{w}']).round(6)
        if {'loghours_3', 'loghours_4'} <= set(imp.columns):
            frame['d_hours'] = frame['hours_4'] - frame['hours_3']
        datasets.append(frame)
    return datasets


def pool(estimates, variances, n, k):
    '''Правила Рубина для одного коэффициента; степени свободы по Барнарду — Рубину.'''
    q, u = np.asarray(estimates), np.asarray(variances)
    m = len(q)
    q_bar, within, between = q.mean(), u.mean(), q.var(ddof=1)
    total = within + (1 + 1 / m) * between
    lam = (1 + 1 / m) * between / total
    df_old = (m - 1) / lam ** 2 if lam > 0 else np.inf
    df_obs = (n - k + 1) / (n - k + 3) * (n - k) * (1 - lam)
    df = 1 / (1 / df_old + 1 / df_obs)
    t = stats.t.ppf(0.975, df)
    return {'оценка': q_bar, 'SE': np.sqrt(total), 'ДИ 95%: от': q_bar - t * np.sqrt(total),
            'ДИ 95%: до': q_bar + t * np.sqrt(total), 'df': df,
            'p': 2 * stats.t.sf(abs(q_bar / np.sqrt(total)), df), 'FMI': lam}


def fit_pooled(formula, set_name, rows=None, model=smf.ols, **fit_kwargs):
    '''Оценивает модель на каждом восстановленном наборе и объединяет коэффициенты.
    rows — функция frame -> булева маска (условие выборки внутри набора).'''
    if model is not smf.ols:
        fit_kwargs.setdefault('disp', 0)
    fits = []
    for frame in imputed_datasets(set_name):
        if rows is not None:
            frame = frame[rows(frame)]
        fits.append(model(formula, frame).fit(**fit_kwargs))
    n, k = int(np.mean([f.nobs for f in fits])), len(fits[0].params)
    return pd.DataFrame({name: pool([f.params[name] for f in fits], [f.bse[name] ** 2 for f in fits], n, k)
                         for name in fits[0].params.index}).T.assign(n=n, m=len(fits))


pd.DataFrame({name: {'записей': long['row'].nunique(), 'm': long['.imp'].max()} for name, long in IMPUTED.items()}).T

,записей,m
w3,1204,49
w4,904,52
work3,531,60
work4,438,62
intern3,241,57
nw3,338,41
workboth,153,44


**Вывод.** Все 7 наборов загружены, в каждом от 41 до 62 вариантов. Пример вызова:

```python
fit_pooled('sat_4 ~ hours_4 + sex_c + major_c + funding_c + ses', 'work4',
           rows=lambda f: f['sat_4'].notna())
```
- `rows` задаёт выборку внутри набора; записи без значения зависимой переменной нужно исключать именно здесь.
- Для логистической модели — `model=smf.logit`.
- Проверка устойчивости на восстановленных данных: добавьте в `rows` условие по флагу, например `d.loc[f.index, 'q_hours_4'].isin([0, 1, 2])`.

## 7. Как проверяется каждая гипотеза

Правила — `analysis_plan.md`, раздел 2. Функция `test()` оценивает одну проверку в четырёх вариантах:

| Вариант | Данные |
|---|---|
| **A — основной** | восстановленные наборы, правила Рубина |
| B | полные случаи |
| C | полные случаи без материального положения |
| D | восстановленные наборы без сомнительных значений (флаги 3–5) |

Тип модели задаётся типом зависимой переменной: логит (да/нет), порядковая логистическая (шкалы 1–4, 1–5), МНК с ошибками HC3 (сон). Для логита и порядковой модели эффект — отношение шансов (OR): > 1 — шансы более высокого значения растут, < 1 — снижаются. Для МНК — изменение в часах сна.

`report()` добавляет поправку Холма внутри каждой гипотезы (по волнам и дополнительным контрастам) и проверяет направление. `show()` — компактная таблица: оценка [95% ДИ], p для каждого варианта. Метки в названии проверки: «доп.» — дополнительный контраст (входит в поправку Холма, не входит в решение), «устойч.» — проверка устойчивости (не входит ни в поправку, ни в решение).

В таблицах p для каждого варианта — без поправки; «p Холм (A)» — с поправкой. «Вывод устойчив» — в вариантах B–D (с той же поправкой Холма) вывод тот же, что в A.

«Поправки сокращены» — в выборке основного варианта меньше 10 событий (логит) или наблюдений на параметр, поэтому из поправок оставлены только пол и направление (`analysis_plan.md`, 2.4).

Столбец «OR по порогам (B)» — проверка пропорциональности шансов: OR в отдельных моделях «≥ 2», «≥ 3», «≥ 4». Близкие значения — предположение порядковой модели выполняется.

In [7]:
CTRL = {'sex': "C(sex_c, Treatment('мужской'))", 'major': "C(major_c, Treatment('инженерные'))",
        'funding': "C(funding_c, Treatment('бюджет'))", 'ses': 'ses'}
SOCIAL = ['sex', 'major', 'funding', 'ses']
FLAG_FREE = {'sex_c', 'major_c', 'funding_c', 'ses'}  # флаги контролей проверяются отдельно


def prepare(frame):
    '''Переменные для моделей: часы на 10, фиктивные форматы, категории — строками.'''
    frame = frame.copy()
    for w in ['3', '4']:
        frame[f'h10_{w}'] = frame[f'hours_{w}'] / 10
        fmt = frame[f'fmt_{w}'].astype(object)
        for code, label in [('full', 'полный день'), ('part', 'неполный день'), ('odd', 'подработки')]:
            frame[f'{code}_{w}'] = fmt.eq(label).astype(float).where(fmt.notna())
    frame['d_h10'] = frame['d_hours'] / 10
    frame['paid'] = frame['funding_c'].astype(object).eq('платное').astype(float).where(frame['funding_c'].notna())
    for w in ['3', '4']:
        motive = frame[f'motive_g14_{w}']
        frame[f'self_{w}'] = motive.eq('содержать себя').astype(float).where(motive.notna())
    frame['self_rec_4'] = frame['motive_g14_rec_4'].eq('содержать себя').astype(float).where(frame['motive_g14_rec_4'].notna())
    frame['alloc10_3'] = frame['alloc_работа_3'] * 10
    for c in ['sex_c', 'major_c', 'funding_c', 'motive_g35_4', 'intern_motive_g44_3', 'motive_raw_3', 'motive_raw_4',
              'work_intern_3', 'work_intern_4', 'motive_g35_rec_4']:
        frame[c] = frame[c].astype(object).where(frame[c].notna())
    return frame


D = prepare(d)
# переменные модели → переменные с флагами качества
VAR_OF = {'h10_3': ['hours_3'], 'h10_4': ['hours_4'], 'd_h10': ['hours_3', 'hours_4'], 'paid': ['funding_c'],
          'd_sleep_wd': ['sleep_dur_wd_3', 'sleep_dur_wd_4'], 'self_3': ['motive_g14_3'], 'self_4': ['motive_g14_4'],
          'self_rec_4': ['motive_g14_4'], 'alloc10_3': ['alloc_работа_3']}


def flag_names(col):
    return {quality_name(v) for v in VAR_OF.get(col, [col])} - {None}


def variables(formula):
    '''Имена столбцов, участвующих в формуле.'''
    names = set()
    for token in patsy.ModelDesc.from_formula(formula).rhs_termlist + patsy.ModelDesc.from_formula(formula).lhs_termlist:
        for factor in token.factors:
            code = factor.name()
            names |= {c for c in D.columns if re.search(rf'(?<![\w]){re.escape(c)}(?![\w])', code)}
    return names


def fit(kind, formula, frame):
    '''Оценка модели; возвращает результат statsmodels.'''
    if kind == 'logit':
        return smf.logit(formula, frame).fit(disp=0)
    if kind == 'ols':
        return smf.ols(formula, frame).fit(cov_type='HC3')
    y, X = patsy.dmatrices(formula, frame, return_type='dataframe')
    X = X.drop(columns='Intercept')
    return OrderedModel(y.iloc[:, 0].astype(int), X, distr='logit').fit(method='bfgs', disp=0, maxiter=2000)


def flagged_ok(frame, cols, robust):
    '''Строки, где у переменных модели допустимые флаги: 0–5 (основной) или 0–2 (без сомнительных).'''
    ok = pd.Series(True, index=frame.index)
    allowed = [0, 1, 2] if robust else [0, 1, 2, 3, 4, 5]
    for col in cols:
        for name in flag_names(col):
            ok &= d.loc[frame.index, f'q_{name}'].isin(allowed)
    return ok


def complete_frame(formula, rows, robust=False):
    frame = D[rows(D)]
    cols = variables(formula)
    frame = frame[frame[list(cols)].notna().all(axis=1)]
    return frame[flagged_ok(frame, cols - FLAG_FREE, robust)]


def imputed_frames(formula, set_name, rows, robust=False):
    cols = variables(formula)
    frames = []
    for frame in imputed_datasets(set_name):
        frame = prepare(frame)
        frame = frame[rows(frame)]
        frame = frame[frame[list(cols)].notna().all(axis=1)]
        if robust:  # восстановленные значения (код 2) допустимы; исключаются коды 3–5
            ok = pd.Series(True, index=frame.index)
            for col in cols - FLAG_FREE:
                for name in flag_names(col):
                    ok &= d.loc[frame.index, f'q_{name}'].isin([0, 1, 2])
            frame = frame[ok]
        frames.append(frame)
    return frames


def n_params(formula, frame):
    return patsy.dmatrices(formula, frame, return_type='dataframe')[1].shape[1] - 1


def enough(kind, formula, frame):
    '''Правило численности: 10 событий (логит) или 10 наблюдений (остальные) на параметр.'''
    y = patsy.dmatrices(formula, frame, return_type='dataframe')[0].iloc[:, 0]
    base = min(y.sum(), len(y) - y.sum()) if kind == 'logit' else len(y)
    return base / n_params(formula, frame) >= 10


def estimate(kind, formula, frames, term):
    '''Оценка term по одному набору (полные случаи) или нескольким (правила Рубина).'''
    results = [fit(kind, formula, f) for f in frames]
    if len(results) == 1:
        r = results[0]
        b, se, p, n = r.params[term], r.bse[term], r.pvalues[term], int(r.nobs)
        ci = r.conf_int().loc[term].to_numpy()
        return {'n': n, 'b': b, 'lo': ci[0], 'hi': ci[1], 'p': p}
    n = int(np.mean([r.nobs for r in results]))
    k = len(results[0].params)
    pooled = pool([r.params[term] for r in results], [r.bse[term] ** 2 for r in results], n, k)
    return {'n': n, 'b': pooled['оценка'], 'lo': pooled['ДИ 95%: от'], 'hi': pooled['ДИ 95%: до'], 'p': pooled['p'],
            'FMI': pooled['FMI']}


def threshold_or(formula, frame, term):
    '''Проверка пропорциональности шансов: OR term в бинарных моделях y ≥ k.'''
    y_name = formula.split('~')[0].strip()
    values = sorted(frame[y_name].dropna().unique())[1:]
    out = []
    for k in values:
        f = frame.assign(_y=(frame[y_name] >= k).astype(int))
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                r = smf.logit(formula.replace(y_name, '_y', 1), f).fit(disp=0, maxiter=300)
            ok = np.isfinite(r.params[term]) and abs(r.params[term]) < 10  # разделение по поправкам не мешает оценке term
            out.append(f'≥{k:g}: {np.exp(r.params[term]):.2f}' if ok else f'≥{k:g}: —')
        except Exception:
            out.append(f'≥{k:g}: —')
    return '; '.join(out)  # «—»: оценка неустойчива — на этом пороге слишком мало ответов


def test(hyp, label, kind, y, terms, controls, rows, set_name, direction, focal=None):
    '''Одна проверка в четырёх вариантах. terms — независимые и дополнительные поправки; focal — оцениваемый член.'''
    focal = focal or terms[0]
    social = [CTRL[c] for c in controls]
    formula = f'{y} ~ ' + ' + '.join(terms + social)
    reduced = False
    if not enough(kind, formula, imputed_frames(formula, set_name, rows)[0]):  # численность — по выборке варианта A
        social = [CTRL[c] for c in controls if c in ('sex', 'major')]
        formula, reduced = f'{y} ~ ' + ' + '.join(terms + social), True
    base = complete_frame(formula, rows)
    formula_no_ses = f'{y} ~ ' + ' + '.join(terms + [s for s in social if s != 'ses'])
    row = {'гипотеза': hyp, 'проверка': label, 'модель': {'logit': 'логит', 'ordinal': 'порядковая', 'ols': 'МНК, HC3'}[kind],
           'поправки сокращены': reduced, 'ожидаемый знак': '+' if direction > 0 else '−'}
    variants = {'A': estimate(kind, formula, imputed_frames(formula, set_name, rows), focal),
                'B': estimate(kind, formula, [base], focal),
                'C': (estimate(kind, formula_no_ses, [complete_frame(formula_no_ses, rows)], focal)
                      if 'ses' not in terms else {'n': np.nan, 'b': np.nan, 'lo': np.nan, 'hi': np.nan, 'p': np.nan}),
                'D': estimate(kind, formula, imputed_frames(formula, set_name, rows, robust=True), focal)}
    for v, res in variants.items():
        row.update({f'{v}: {k}': val for k, val in res.items()})
    if kind == 'ordinal':
        row['OR по порогам (B)'] = threshold_or(formula, base, focal)
    return row


def holm(table, column):
    '''Поправка Холма внутри каждой гипотезы; проверки устойчивости («устойч.») в неё не входят.'''
    out = pd.Series(np.nan, index=table.index)
    counted = ~table['проверка'].str.contains('устойч.')
    for _, g in table[counted].groupby('гипотеза'):
        out[g.index] = multipletests(g[column], method='holm')[1]
    return out


def report(rows):
    '''Таблица результатов с поправкой Холма внутри каждой гипотезы и решением.
    Метки в названии проверки: «доп.» — дополнительный контраст (входит в поправку Холма, но не в решение);
    «устойч.» — проверка устойчивости (не входит ни в поправку, ни в решение).'''
    table = pd.DataFrame(rows)
    sign = np.where(table['ожидаемый знак'] == '+', 1, -1)
    ratio = (table['модель'] != 'МНК, HC3').to_numpy()

    def supported(v):
        '''Оценка (на шкале логарифма для OR) в ожидаемую сторону и p (Холм) < 0,05 в варианте v.'''
        return (np.sign(table[f'{v}: b']) == sign) & (table[f'{v}: p (Холм)'] < 0.05)

    for v in 'ABCD':
        table[f'{v}: p (Холм)'] = holm(table, f'{v}: p').fillna(table[f'{v}: p'])  # «устойч.» — без поправки
    table['в ожидаемую сторону'] = np.sign(table['A: b']) == sign
    table['значимо (A, Холм)'] = table['A: p (Холм)'] < 0.05
    main = supported('A')
    table['совпадает в B, C, D'] = np.logical_and.reduce(
        [supported(v).eq(main) | table[f'{v}: p'].isna() for v in 'BCD'])
    for v in 'ABCD':  # OR — из логарифма отношения шансов
        for col in ['b', 'lo', 'hi']:
            table[f'{v}: {col}'] = np.where(ratio, np.exp(table[f'{v}: {col}']), table[f'{v}: {col}'])
    return table


def show(table):
    '''Компактный вид: оценка [ДИ], p для каждого варианта.'''
    out = table[['гипотеза', 'проверка', 'модель', 'ожидаемый знак', 'A: n']].copy()
    name = np.where(table['модель'] != 'МНК, HC3', 'OR', 'b')
    out['эффект'] = name
    for v in 'ABCD':
        out[v] = [f"{r[f'{v}: b']:.2f} [{r[f'{v}: lo']:.2f}; {r[f'{v}: hi']:.2f}], p = {r[f'{v}: p']:.3f}"
                  if pd.notna(r[f'{v}: p']) else '—' for _, r in table.iterrows()]
    out['p Холм (A)'] = table['A: p (Холм)'].round(3)
    out[[f'{v}: n' for v in 'BCD']] = table[[f'{v}: n' for v in 'BCD']]
    out['в ожидаемую сторону'] = table['в ожидаемую сторону']
    out['вывод устойчив'] = table['совпадает в B, C, D']
    extra = [c for c in ['OR по порогам (B)', 'поправки сокращены'] if c in table]
    return pd.concat([out, table[extra]], axis=1).set_index(['гипотеза', 'проверка'])


def verdict(table):
    '''Решение по каждой гипотезе: подтверждается / частично / не подтверждается.'''
    rows = []
    for hyp, g in table.groupby('гипотеза', sort=False):
        main = g[~g['проверка'].str.contains('доп.|устойч.')]
        ok = (main['в ожидаемую сторону'] & main['значимо (A, Холм)'])
        decision = ('подтверждается' if ok.all() else 'частично подтверждается' if ok.any() else 'не подтверждается')
        rows.append({'гипотеза': hyp, 'проверок': len(main), 'подтвердилось': int(ok.sum()), 'решение': decision,
                     'устойчиво в B–D': bool(main['совпадает в B, C, D'].all())})
    return pd.DataFrame(rows).set_index('гипотеза')

In [8]:
def share(series, value=1):
    """Доля value: «x% [ДИ Уилсона] (k/n)»."""
    s = series.dropna()
    k, n = int((s == value).sum()), len(s)
    if n == 0:
        return '—'
    lo, hi = proportion_confint(k, n, method='wilson')
    return f'{k / n:.0%} [{lo:.0%}; {hi:.0%}] ({k}/{n})'


def shares_by(frame, value_col, group_col, value=1):
    """Доли value по группам с ДИ."""
    return frame.groupby(group_col, observed=True)[value_col].apply(lambda s: share(s, value))


def assoc(a, b):
    """χ² (или перестановочный при ожидаемой частоте < 5) и V Крамера для двух категориальных переменных."""
    table = pd.crosstab(a, b)
    table = table.loc[table.sum(axis=1) > 0, table.sum(axis=0) > 0]
    if min(table.shape) < 2:
        return {'n': int(table.values.sum()), 'p': np.nan, 'V Крамера': np.nan, 'метод': '—'}
    result = stats.chi2_contingency(table, correction=False)
    method = 'χ²'
    if result.expected_freq.min() < 5:
        result = stats.chi2_contingency(table, correction=False, method=stats.PermutationMethod(n_resamples=20_000, rng=rng))
        method = 'перестановки'
    n = table.values.sum()
    return {'n': int(n), 'p': result.pvalue, 'V Крамера': np.sqrt(result.statistic / (n * (min(table.shape) - 1))),
            'метод': method}


def median_iqr(s):
    s = s.dropna()
    return f'{s.median():.1f} ({s.quantile(.25):.1f}–{s.quantile(.75):.1f}), n={len(s)}' if len(s) else '—'


def mean_ci(s):
    s = s.dropna()
    if len(s) < 2:
        return '—'
    half = stats.t.ppf(0.975, len(s) - 1) * s.std() / np.sqrt(len(s))
    return f'{s.mean():.2f} [{s.mean() - half:.2f}; {s.mean() + half:.2f}], n={len(s)}'


FORMAT_ORDER = ['нет работы', 'полный день', 'неполный день', 'подработки']
REASONS = {1: 'разочарование в вузе', 2: 'разочарование в направлении', 3: 'академические трудности',
           4: 'отношения с одногруппниками', 5: 'отношения с преподавателями', 6: 'экономические трудности семьи',
           7: 'трудности совмещения с работой'}
SLEEP_ITEMS = {'3': ('sleep1_3w', 'sleep2_3w', 'sleep3_3w', 'sleep4_3w'),
               '4': ('sleep1_4w', 'sleep2_4w', 'sleep3_4w', 'sleep4_4w')}
ALLOC_COLS = ['alloc_учёба_3', 'alloc_клубы_3', 'alloc_работа_3', 'alloc_волонтёрство_3', 'alloc_курсы_3', 'alloc_остальное_3']


def tertiles(s):
    """Терцили рабочих часов: метки с границами."""
    return pd.qcut(s, 3, duplicates='drop')

**Вывод.** Функции готовы; ниже они применяются к каждой гипотезе по порядку `hypotheses.md`.

## 8. ИВ1. Структура и социальные различия занятости

### З1.1. Структура занятости

Доли форматов среди ответивших о работе; среди работающих — часы, сферы и мотивы.

In [9]:
structure = pd.DataFrame({f'{w}-я волна': {fmt: share(d.loc[d[f'in_w{w}'], f'fmt_{w}'].astype(object), fmt)
                                           for fmt in FORMAT_ORDER} for w in ['3', '4']})
display(structure.rename_axis('формат'))
hours_by_fmt = pd.DataFrame({f'{w}-я волна': d[d[f'emp_{w}'].eq(1)].groupby(f'fmt_{w}', observed=True)[f'hours_{w}']
                             .apply(median_iqr) for w in ['3', '4']})
display(hours_by_fmt.rename_axis('часы: медиана (IQR)'))
workers = {w: d[d[f'emp_{w}'].eq(1)] for w in ['3', '4']}
display(pd.DataFrame({f'{w}-я волна': {s: share(workers[w][f'sphere_{w}'].astype(object), s)
                                       for s in d['sphere_4'].cat.categories} for w in ['3', '4']}).rename_axis('сфера'))
pd.DataFrame({f'{w}-я волна': {m: share(workers[w][f'motive_raw_{w}'].astype(object), m)
                               for m in d['motive_raw_4'].cat.categories} for w in ['3', '4']}).rename_axis('мотив')

,3-я волна,4-я волна
формат,,
нет работы,56% [53%; 59%] (678/1211),52% [49%; 55%] (473/914)
полный день,4% [3%; 5%] (49/1211),9% [7%; 11%] (84/914)
неполный день,21% [19%; 23%] (254/1211),20% [18%; 23%] (186/914)
подработки,19% [17%; 21%] (230/1211),19% [16%; 21%] (171/914)


,3-я волна,4-я волна
часы: медиана (IQR),,
полный день,"34.5 (20.0–45.8), n=44","35.0 (20.0–40.0), n=78"
неполный день,"20.0 (10.0–26.0), n=233","20.0 (12.0–25.0), n=181"
подработки,"9.0 (5.0–15.0), n=177","7.5 (0.0–15.2), n=168"


,3-я волна,4-я волна
сфера,,
образование,22% [19%; 27%] (96/429),16% [13%; 20%] (71/440)
наука,12% [10%; 16%] (53/429),11% [9%; 15%] (50/440)
бизнес: наёмный работник,26% [22%; 31%] (113/429),30% [26%; 34%] (132/440)
бизнес: предприниматель,4% [2%; 6%] (16/429),4% [2%; 6%] (16/440)
госуправление,4% [2%; 6%] (17/429),2% [1%; 4%] (8/440)
общественные и политические организации,2% [1%; 4%] (8/429),3% [2%; 5%] (13/440)
СМИ,4% [2%; 6%] (17/429),2% [1%; 4%] (11/440)
фриланс,25% [22%; 30%] (109/429),19% [15%; 23%] (83/440)
другое,0% [0%; 1%] (0/429),13% [10%; 16%] (56/440)


,3-я волна,4-я волна
мотив,,
оплата обучения,2% [1%; 4%] (12/533),3% [2%; 5%] (12/440)
содержать себя,34% [30%; 38%] (179/533),38% [34%; 43%] (169/440)
доп. доход,34% [30%; 38%] (183/533),28% [24%; 33%] (124/440)
опыт,16% [13%; 20%] (87/533),19% [15%; 23%] (82/440)
интерес,11% [8%; 13%] (56/533),8% [5%; 10%] (33/440)
другое,3% [2%; 5%] (16/533),5% [3%; 7%] (20/440)


**Вывод.** Не работают 56% участников 3-й волны и 52% — 4-й. Работают полный день 4% и 9%, неполный день — 21% и 20%, на подработках — 19% в обеих волнах. Медиана часов: полный день — 35, неполный — 20, подработки — 8–9. Самые частые сферы — наёмная работа в бизнесе (26% и 30%), фриланс (25% и 19%) и образование (22% и 16%). Самые частые мотивы — «содержать себя» (34% и 38%) и «доп. доход» (34% и 28%); «оплату обучения» называют 2–3%.

### Г1.2. Платное обучение и наличие работы

Логистическая регрессия: работа (1/0) ~ платное против бюджета + пол, направление, материальное положение. Целевое обучение исключено из модели.

In [10]:
rows = [test('Г1.2', f'{w}-я волна', 'logit', f'emp_{w}', ['paid'], ['sex', 'major', 'ses'],
             lambda f, w=w: f[f'in_w{w}'] & f['funding_c'].isin(['бюджет', 'платное']), f'w{w}', +1) for w in ['3', '4']]
ALL_ROWS = list(rows)
display(show(report(rows)))
pd.DataFrame({f'{w}-я волна': {'целевое': share(d.loc[d[f'in_w{w}'] & d['funding_c'].eq('целевое'), f'emp_{w}'])}
              for w in ['3', '4']}).rename_axis('доля работающих')

модель ожидаемый знак  A: n эффект  \
гипотеза проверка                                       
Г1.2     3-я волна  логит              +  1151     OR   
         4-я волна  логит              +   876     OR   

                                               A  \
гипотеза проверка                                  
Г1.2     3-я волна  1.00 [0.69; 1.46], p = 0.987   
         4-я волна  0.93 [0.60; 1.45], p = 0.745   

                                               B  \
гипотеза проверка                                  
Г1.2     3-я волна  0.92 [0.60; 1.42], p = 0.718   
         4-я волна  0.94 [0.59; 1.50], p = 0.811   

                                               C  \
гипотеза проверка                                  
Г1.2     3-я волна  0.98 [0.66; 1.46], p = 0.933   
         4-я волна  0.96 [0.62; 1.48], p = 0.851   

                                               D  p Холм (A)  B: n  C: n  \
гипотеза проверка                                                          
Г1.2     3-я волна  1.00 [0.69; 1.46], p = 0.987         1.0   742   871   
         4-я волна  0.93 [0.60; 1.45], p = 0.745         1.0   560   653   

                    D: n  в ожидаемую сторону  вывод устойчив  \
гипотеза проверка                                               
Г1.2     3-я волна  1151                 True            True   
         4-я волна   876                False            True   

                    поправки сокращены  
гипотеза проверка                       
Г1.2     3-я волна               False  
         4-я волна               False

,3-я волна,4-я волна
доля работающих,,
целевое,50% [35%; 65%] (20/40),71% [47%; 87%] (12/17)


**Вывод. Г1.2 не подтверждается.** Студенты платной формы работают так же часто, как бюджетники: OR = 1,00 в 3-й волне и 0,93 в 4-й, доверительные интервалы широко накрывают 1. Вывод одинаков во всех четырёх вариантах. Среди целевиков работают 50% и 71%, но это 40 и 17 человек — оценка очень неточная.

### Г1.3. Материальное положение и наличие работы

Логистическая регрессия: работа ~ материальное положение (1–6) + финансирование, пол, направление.

In [11]:
rows = [test('Г1.3', f'{w}-я волна', 'logit', f'emp_{w}', ['ses'], ['sex', 'major', 'funding'],
             lambda f, w=w: f[f'in_w{w}'], f'w{w}', -1) for w in ['3', '4']]
ALL_ROWS += rows
show(report(rows))

модель ожидаемый знак  A: n эффект  \
гипотеза проверка                                       
Г1.3     3-я волна  логит              −  1204     OR   
         4-я волна  логит              −   904     OR   

                                               A  \
гипотеза проверка                                  
Г1.3     3-я волна  0.99 [0.87; 1.12], p = 0.826   
         4-я волна  1.07 [0.91; 1.26], p = 0.387   

                                               B  C  \
гипотеза проверка                                     
Г1.3     3-я волна  0.99 [0.87; 1.12], p = 0.841  —   
         4-я волна  1.06 [0.90; 1.25], p = 0.494  —   

                                               D  p Холм (A)  B: n  C: n  \
гипотеза проверка                                                          
Г1.3     3-я волна  0.99 [0.87; 1.12], p = 0.826       0.826   776   NaN   
         4-я волна  1.07 [0.91; 1.26], p = 0.387       0.775   573   NaN   

                    D: n  в ожидаемую сторону  вывод устойчив  \
гипотеза проверка                                               
Г1.3     3-я волна  1204                 True            True   
         4-я волна   904                False            True   

                    поправки сокращены  
гипотеза проверка                       
Г1.3     3-я волна               False  
         4-я волна               False

**Вывод. Г1.3 не подтверждается.** Материальное положение семьи не связано с наличием работы: OR = 0,99 и 1,07 на одну ступень шкалы, p > 0,3. Вывод устойчив.

### Г1.4. Материальное положение и мотив «содержать себя»

Логистическая регрессия среди работающих, выбравших «содержать себя» (1) или «опыт и интерес» (0). Проверка устойчивости — с перекодированными открытыми ответами.

In [12]:
rows = [test('Г1.4', f'{w}-я волна', 'logit', f'self_{w}', ['ses'], ['sex', 'major', 'funding'],
             lambda f, w=w: f[f'in_w{w}'] & f[f'emp_{w}'].eq(1), f'work{w}', -1) for w in ['3', '4']]
rows.append(test('Г1.4', 'устойч.: 4-я волна, открытые ответы перекодированы', 'logit', 'self_rec_4', ['ses'],
                 ['sex', 'major', 'funding'], lambda f: f['in_w4'] & f['emp_4'].eq(1), 'work4', -1))
ALL_ROWS += rows
display(show(report(rows)))
pd.concat({f'{w}-я волна': pd.crosstab(d[f'motive_g14_{w}'], d['ses4']) for w in ['3', '4']}).rename_axis(['волна', 'мотив'])

модель  \
гипотеза проверка                                                    
Г1.4     3-я волна                                           логит   
         4-я волна                                           логит   
         устойч.: 4-я волна, открытые ответы перекодированы  логит   

                                                            ожидаемый знак  \
гипотеза проверка                                                            
Г1.4     3-я волна                                                       −   
         4-я волна                                                       −   
         устойч.: 4-я волна, открытые ответы перекодированы              −   

                                                             A: n эффект  \
гипотеза проверка                                                          
Г1.4     3-я волна                                            516     OR   
         4-я волна                                            417     OR   
         устойч.: 4-я волна, открытые ответы перекодированы   421     OR   

                                                                                        A  \
гипотеза проверка                                                                           
Г1.4     3-я волна                                           0.89 [0.75; 1.05], p = 0.175   
         4-я волна                                           0.92 [0.72; 1.17], p = 0.507   
         устойч.: 4-я волна, открытые ответы перекодированы  0.92 [0.72; 1.18], p = 0.512   

                                                                                        B  \
гипотеза проверка                                                                           
Г1.4     3-я волна                                           0.83 [0.69; 1.00], p = 0.046   
         4-я волна                                           0.90 [0.69; 1.17], p = 0.427   
         устойч.: 4-я волна, открытые ответы перекодированы  0.90 [0.69; 1.17], p = 0.424   

                                                             C  \
гипотеза проверка                                                
Г1.4     3-я волна                                           —   
         4-я волна                                           —   
         устойч.: 4-я волна, открытые ответы перекодированы  —   

                                                                                        D  \
гипотеза проверка                                                                           
Г1.4     3-я волна                                           0.89 [0.75; 1.05], p = 0.175   
         4-я волна                                           0.94 [0.73; 1.19], p = 0.588   
         устойч.: 4-я волна, открытые ответы перекодированы  0.94 [0.73; 1.19], p = 0.593   

                                                             p Холм (A)  B: n  \
гипотеза проверка                                                               
Г1.4     3-я волна                                                0.350   337   
         4-я волна                                                0.507   253   
         устойч.: 4-я волна, открытые ответы перекодированы       0.512   254   

                                                             C: n  D: n  \
гипотеза проверка                                                         
Г1.4     3-я волна                                            NaN   516   
         4-я волна                                            NaN   410   
         устойч.: 4-я волна, открытые ответы перекодированы   NaN   414   

                                                             в ожидаемую сторону  \
гипотеза проверка                                                                  
Г1.4     3-я волна                                                          True   
         4-я волна                                                          True   
         устойч.: 4-я волна, открытые ответы перекодированы                 True   

     

ses4                       1–3   4   5   6
волна     мотив                           
3-я волна доп. доход        16  22  54  38
          оплата обучения    2   0   4   0
          опыт и интерес    11   6  46  36
          содержать себя    19  19  42  27
4-я волна доп. доход         4  11  44  17
          оплата обучения    2   2   2   2
          опыт и интерес     7   9  37  25
          содержать себя     9  23  35  30

**Вывод. Г1.4 не подтверждается.** Направление совпадает с гипотезой — при лучшем положении семьи «содержать себя» называют чуть реже (OR = 0,89 и 0,92), — но эффект не значим (p по Холму 0,35 и 0,51). Только в полных случаях 3-й волны p = 0,046 без поправки; с поправкой Холма и в остальных вариантах значимости нет. Перекодировка открытых ответов результат не меняет.

### З1.5. Мотив «оплата обучения» и финансирование

In [13]:
z15 = pd.DataFrame({f'{w}-я волна': shares_by(d[d[f'emp_{w}'].eq(1)], f'pay_tuition_{w}', 'funding_c') for w in ['3', '4']})
display(z15.rename_axis('финансирование'))
pd.DataFrame({f'{w}-я волна': assoc(d.loc[d[f'emp_{w}'].eq(1), 'funding_c'], d.loc[d[f'emp_{w}'].eq(1), f'pay_tuition_{w}'])
              for w in ['3', '4']})

,3-я волна,4-я волна
финансирование,,
бюджет,1% [0%; 3%] (3/316),3% [1%; 6%] (7/244)
платное,4% [1%; 11%] (3/74),2% [0%; 9%] (1/60)
целевое,0% [0%; 16%] (0/20),0% [0%; 24%] (0/12)


,3-я волна,4-я волна
n,410,316
p,0.102245,0.787861
V Крамера,0.102651,0.043794
метод,перестановки,перестановки


**Вывод.** Мотив «оплата обучения» редок во всех группах: 1–4% у бюджетников и платников, 0 у целевиков. Различия по финансированию не значимы (p = 0,10 и 0,79). Даже у платников это мотив не более чем 4%: студенты платной формы, по-видимому, работают не ради оплаты учёбы.

### З1.6. Различия между сферами работы

In [14]:
for w in ['3', '4']:
    wk = d[d[f'emp_{w}'].eq(1) & d[f'sphere_{w}'].notna()]
    profile = wk.groupby(f'sphere_{w}', observed=True).agg(
        n=(f'fmt_{w}', 'size'), **{'медиана часов': (f'hours_{w}', 'median'),
                                   'полный день, %': (f'fmt_{w}', lambda s: s.eq('полный день').mean() * 100),
                                   'подработки, %': (f'fmt_{w}', lambda s: s.eq('подработки').mean() * 100),
                                   'частый мотив': (f'motive_raw_{w}', lambda s: s.mode().iloc[0] if s.notna().any() else '—')})
    if w == '4':
        profile['соответствие специальности'] = wk.groupby('sphere_4', observed=True)['fit_4'].mean()
    display(profile.round(1).rename_axis(f'сфера, {w}-я волна'))
    big = wk[wk.groupby(f'sphere_{w}', observed=True)[f'fmt_{w}'].transform('size') >= 10]
    tests = {'формат': assoc(big[f'sphere_{w}'].astype(str), big[f'fmt_{w}'].astype(str)),
             'мотив': assoc(big[f'sphere_{w}'].astype(str), big[f'motive_raw_{w}'].astype(str)),
             'часы (Краскел — Уоллис)': {'n': int(big[f'hours_{w}'].notna().sum()),
                                         'p': stats.kruskal(*[g.dropna() for _, g in big.groupby(f'sphere_{w}', observed=True)[f'hours_{w}']
                                                              if g.notna().sum() > 0]).pvalue}}
    if w == '4':
        tests['соответствие (Краскел — Уоллис)'] = {'n': int(big['fit_4'].notna().sum()), 'p': stats.kruskal(
            *[g.dropna() for _, g in big.groupby('sphere_4', observed=True)['fit_4'] if g.notna().sum() > 0]).pvalue}
    display(pd.DataFrame(tests).T.rename_axis(f'сферы с n ≥ 10, {w}-я волна'))

,n,медиана часов,"полный день, %","подработки, %",частый мотив
"сфера, 3-я волна",,,,,
образование,96,10.0,7.3,39.6,содержать себя
наука,53,16.0,11.3,24.5,опыт
бизнес: наёмный работник,113,20.0,13.3,27.4,содержать себя
бизнес: предприниматель,16,14.0,18.8,56.2,интерес
госуправление,17,20.0,5.9,35.3,доп. доход
общественные и политические организации,8,29.0,12.5,50.0,доп. доход
СМИ,17,14.5,5.9,29.4,содержать себя
фриланс,109,12.0,5.5,71.6,доп. доход


,n,p,V Крамера,метод
"сферы с n ≥ 10, 3-я волна",,,,
формат,421,0.00005,0.274412,перестановки
мотив,421,0.00095,0.17616,перестановки
часы (Краскел — Уоллис),359.0,0.00001,NaN,NaN


,n,медиана часов,"полный день, %","подработки, %",частый мотив,соответствие специальности
"сфера, 4-я волна",,,,,,
образование,71,12.0,18.3,31.0,содержать себя,2.6
наука,50,16.0,14.0,22.0,опыт,3.3
бизнес: наёмный работник,132,20.0,25.0,28.8,содержать себя,2.5
бизнес: предприниматель,16,20.0,25.0,43.8,доп. доход,1.9
госуправление,8,20.0,50.0,12.5,содержать себя,3.0
общественные и политические организации,13,35.0,53.8,23.1,опыт,2.3
СМИ,11,20.0,18.2,45.5,интерес,1.7
фриланс,83,10.0,7.2,69.9,содержать себя,2.3
другое,56,19.0,14.3,44.6,содержать себя,1.9


,n,p,V Крамера,метод
"сферы с n ≥ 10, 4-я волна",,,,
формат,432,0.00005,0.275016,перестановки
мотив,432,0.00015,0.199481,перестановки
часы (Краскел — Уоллис),421.0,0.00366,NaN,NaN
соответствие (Краскел — Уоллис),419.0,0.0,NaN,NaN


**Вывод.** Сферы заметно различаются по формату (V Крамера = 0,27, p < 0,001), мотивам (V = 0,18–0,20), часам (p < 0,01) и — в 4-й волне — по соответствию специальности (p < 0,001). Фриланс — в основном подработки (70%); наука — самое высокое соответствие специальности (3,3 из 4), СМИ и предпринимательство — самое низкое (1,7–1,9). Сферы с числом ответов меньше 10 (госуправление, общественные организации в 4-й волне) показаны, но в тесты не входили.

## 9. ИВ2. Занятость и обучение

### Г2.1. Рабочие часы и категория оценок

Порядковая логистическая регрессия: оценки (1–4) ~ часы (на 10 ч) + общий набор поправок. Среди работающих.

In [15]:
rows = [test('Г2.1', f'{w}-я волна', 'ordinal', f'mark_{w}', [f'h10_{w}'], SOCIAL,
             lambda f, w=w: f[f'in_w{w}'] & f[f'emp_{w}'].eq(1), f'work{w}', -1) for w in ['3', '4']]
ALL_ROWS += rows
show(report(rows))

модель ожидаемый знак  A: n эффект  \
гипотеза проверка                                            
Г2.1     3-я волна  порядковая              −   514     OR   
         4-я волна  порядковая              −   421     OR   

                                               A  \
гипотеза проверка                                  
Г2.1     3-я волна  1.05 [0.96; 1.15], p = 0.300   
         4-я волна  0.97 [0.86; 1.09], p = 0.562   

                                               B  \
гипотеза проверка                                  
Г2.1     3-я волна  1.08 [0.96; 1.23], p = 0.202   
         4-я волна  1.07 [0.92; 1.25], p = 0.368   

                                               C  \
гипотеза проверка                                  
Г2.1     3-я волна  1.06 [0.95; 1.20], p = 0.298   
         4-я волна  1.05 [0.92; 1.20], p = 0.491   

                                               D  p Холм (A)  B: n  C: n  \
гипотеза проверка                                                          
Г2.1     3-я волна  1.02 [0.90; 1.16], p = 0.794       0.599   281   329   
         4-я волна  0.94 [0.80; 1.10], p = 0.439       0.599   245   292   

                    D: n  в ожидаемую сторону  вывод устойчив  \
гипотеза проверка                                               
Г2.1     3-я волна   490                False            True   
         4-я волна   345                 True            True   

                               OR по порогам (B)  поправки сокращены  
гипотеза проверка                                                     
Г2.1     3-я волна  ≥2: 1.09; ≥3: 1.09; ≥4: 1.10               False  
         4-я волна  ≥2: 1.27; ≥3: 1.10; ≥4: 1.00               False

**Вывод. Г2.1 не подтверждается.** Число рабочих часов не связано с категорией оценок: OR = 1,05 и 0,97 на 10 часов, p > 0,3. Вывод одинаков во всех вариантах. Отношения шансов по порогам близки — предположение порядковой модели выполняется.

### Г2.2. Рабочие часы и удовлетворённость обучением

In [16]:
rows = [test('Г2.2', f'{w}-я волна', 'ordinal', f'sat_{w}', [f'h10_{w}'], SOCIAL,
             lambda f, w=w: f[f'in_w{w}'] & f[f'emp_{w}'].eq(1), f'work{w}', -1) for w in ['3', '4']]
ALL_ROWS += rows
display(show(report(rows)))
print('«Затрудняюсь» в удовлетворённости среди работающих 4-й волны:',
      int((d['emp_4'].eq(1) & d['satisfaction_4w_st'].eq('DK')).sum()))

модель ожидаемый знак  A: n эффект  \
гипотеза проверка                                            
Г2.2     3-я волна  порядковая              −   505     OR   
         4-я волна  порядковая              −   400     OR   

                                               A  \
гипотеза проверка                                  
Г2.2     3-я волна  0.93 [0.84; 1.03], p = 0.150   
         4-я волна  0.98 [0.86; 1.13], p = 0.788   

                                               B  \
гипотеза проверка                                  
Г2.2     3-я волна  0.92 [0.81; 1.05], p = 0.226   
         4-я волна  0.88 [0.73; 1.05], p = 0.154   

                                               C  \
гипотеза проверка                                  
Г2.2     3-я волна  0.92 [0.81; 1.04], p = 0.166   
         4-я волна  0.88 [0.75; 1.03], p = 0.121   

                                               D  p Холм (A)  B: n  C: n  \
гипотеза проверка                                                          
Г2.2     3-я волна  0.87 [0.75; 1.01], p = 0.062       0.299   273   322   
         4-я волна  1.07 [0.89; 1.29], p = 0.461       0.788   234   276   

                    D: n  в ожидаемую сторону  вывод устойчив  \
гипотеза проверка                                               
Г2.2     3-я волна   482                 True            True   
         4-я волна   327                 True            True   

                               OR по порогам (B)  поправки сокращены  
гипотеза проверка                                                     
Г2.2     3-я волна  ≥2: 0.92; ≥3: 0.93; ≥4: 0.91               False  
         4-я волна  ≥2: 0.75; ≥3: 0.80; ≥4: 0.95               False

«Затрудняюсь» в удовлетворённости среди работающих 4-й волны: 37


**Вывод. Г2.2 не подтверждается.** Удовлетворённость немного снижается с ростом часов (OR = 0,93 и 0,98 на 10 часов), но эффект не значим (p по Холму 0,30 и 0,79) ни в одном варианте. «Затрудняюсь» среди работающих 4-й волны ответили 37 человек — они в шкалу не входят.

### Г2.3. Работа полный день и мысли об отчислении

Основная модель — работающие полный день против неработающих. Дополнительные контрасты — неполный день и подработки против неработающих (модель со всеми форматами).

In [17]:
rows = []
for w in ['3', '4']:
    rows.append(test('Г2.3', f'{w}-я волна', 'logit', f'drop_any_{w}', [f'full_{w}'], SOCIAL,
                     lambda f, w=w: f[f'in_w{w}'] & f[f'fmt_{w}'].isin(['полный день', 'нет работы']), f'w{w}', +1))
    for focal, name in [(f'part_{w}', 'неполный день'), (f'odd_{w}', 'подработки')]:
        rows.append(test('Г2.3', f'доп.: {w}-я волна, {name} против без работы', 'logit', f'drop_any_{w}',
                         [f'full_{w}', f'part_{w}', f'odd_{w}'], SOCIAL, lambda f, w=w: f[f'in_w{w}'], f'w{w}', +1,
                         focal=focal))
ALL_ROWS += rows
display(show(report(rows)))
pd.DataFrame({f'{w}-я волна': shares_by(d[d[f'in_w{w}']], f'drop_any_{w}', f'fmt_{w}') for w in ['3', '4']})

модель  \
гипотеза проверка                                                  
Г2.3     3-я волна                                         логит   
         доп.: 3-я волна, неполный день против без работы  логит   
         доп.: 3-я волна, подработки против без работы     логит   
         4-я волна                                         логит   
         доп.: 4-я волна, неполный день против без работы  логит   
         доп.: 4-я волна, подработки против без работы     логит   

                                                          ожидаемый знак  \
гипотеза проверка                                                          
Г2.3     3-я волна                                                     +   
         доп.: 3-я волна, неполный день против без работы              +   
         доп.: 3-я волна, подработки против без работы                 +   
         4-я волна                                                     +   
         доп.: 4-я волна, неполный день против без работы              +   
         доп.: 4-я волна, подработки против без работы                 +   

                                                           A: n эффект  \
гипотеза проверка                                                        
Г2.3     3-я волна                                          722     OR   
         доп.: 3-я волна, неполный день против без работы  1204     OR   
         доп.: 3-я волна, подработки против без работы     1204     OR   
         4-я волна                                          550     OR   
         доп.: 4-я волна, неполный день против без работы   904     OR   
         доп.: 4-я волна, подработки против без работы      904     OR   

                                                                                      A  \
гипотеза проверка                                                                         
Г2.3     3-я волна                                         1.81 [0.99; 3.32], p = 0.054   
         доп.: 3-я волна, неполный день против без работы  1.30 [0.97; 1.76], p = 0.083   
         доп.: 3-я волна, подработки против без работы     1.60 [1.18; 2.19], p = 0.003   
         4-я волна                                         1.77 [1.09; 2.87], p = 0.021   
         доп.: 4-я волна, неполный день против без работы  1.16 [0.81; 1.65], p = 0.416   
         доп.: 4-я волна, подработки против без работы     1.05 [0.73; 1.53], p = 0.781   

                                                                                      B  \
гипотеза проверка                                                                         
Г2.3     3-я волна                                         2.43 [1.13; 5.20], p = 0.023   
         доп.: 3-я волна, неполный день против без работы  1.34 [0.92; 1.93], p = 0.123   
         доп.: 3-я волна, подработки против без работы     1.62 [1.10; 2.39], p = 0.014   
         4-я волна                                         2.29 [1.27; 4.13], p = 0.006   
         доп.: 4-я волна, неполный день против без работы  1.64 [1.03; 2.60], p = 0.036   
         доп.: 4-я волна, подработки против без работы     0.96 [0.60; 1.56], p = 0.885   

                                                                                      C  \
гипотеза проверка                                                                         
Г2.3     3-я волна                                         2.38 [1.16; 4.89], p = 0.019   
         доп.: 3-я волна, неполный день против без работы  1.24 [0.88; 1.75], p = 0.213   
         доп.: 3-я волна, подработки против без работы     1.76 [1.24; 2.50], p = 0.002   
         4-я волна                                         2.17 [1.26; 3.72], p = 0.005   
         доп.: 4-я волна, неполный день против без работы  1.51 [0.99; 2.32], p = 0.056   
         доп.: 4-я волна, подработки против без работы     1.07 [0.70; 1.66], p = 0.745   

                                                                                      D  \
гипотеза проверка                

,3-я волна,4-я волна
нет работы,40% [37%; 44%] (274/678),39% [35%; 44%] (186/473)
полный день,53% [39%; 66%] (26/49),52% [42%; 63%] (44/84)
неполный день,47% [41%; 53%] (120/254),42% [35%; 49%] (78/186)
подработки,52% [46%; 59%] (120/230),40% [33%; 47%] (68/171)


**Вывод. Г2.3 не подтверждается по правилу решения, хотя направление совпадает.** Работающие полный день задумываются об отчислении чаще неработающих: 53% против 40% в 3-й волне и 52% против 39% в 4-й; OR = 1,81 (p = 0,054) и 1,77 (p = 0,021). После поправки Холма на шесть проверок гипотезы (две волны и четыре дополнительных контраста) p = 0,22 и 0,10. В полных случаях эффект сильнее (OR = 2,43 и 2,29) и остаётся значимым — поэтому вывод помечен как неустойчивый. Группа полного дня мала (49 и 84 человека), доверительные интервалы широкие. Дополнительно: на подработках в 3-й волне об отчислении думают чаще, чем без работы (OR = 1,60, p по Холму = 0,017).

### Г2.4. Рабочие часы и причина «трудности совмещения»

Логистическая регрессия среди работающих, задумывавшихся об отчислении и отметивших хотя бы одну причину.

In [18]:
rows = [test('Г2.4', f'{w}-я волна', 'logit', f'reason7_{w}', [f'h10_{w}'], SOCIAL,
             lambda f, w=w: f[f'in_w{w}'] & f[f'emp_{w}'].eq(1) & f[f'drop_any_{w}'].eq(1) & f[f'reasons_cmp_{w}'],
             f'work{w}', +1) for w in ['3', '4']]
ALL_ROWS += rows
show(report(rows))

модель ожидаемый знак  A: n эффект  \
гипотеза проверка                                       
Г2.4     3-я волна  логит              +   235     OR   
         4-я волна  логит              +   168     OR   

                                               A  \
гипотеза проверка                                  
Г2.4     3-я волна  1.18 [1.01; 1.39], p = 0.038   
         4-я волна  1.24 [1.00; 1.55], p = 0.049   

                                               B  \
гипотеза проверка                                  
Г2.4     3-я волна  1.11 [0.93; 1.31], p = 0.257   
         4-я волна  1.16 [0.89; 1.52], p = 0.280   

                                               C  \
гипотеза проверка                                  
Г2.4     3-я волна  1.13 [0.95; 1.35], p = 0.159   
         4-я волна  1.18 [0.94; 1.47], p = 0.162   

                                               D  p Холм (A)  B: n  C: n  \
гипотеза проверка                                                          
Г2.4     3-я волна  1.34 [1.06; 1.69], p = 0.015       0.077   132   149   
         4-я волна  1.58 [1.13; 2.20], p = 0.008       0.077   100   119   

                    D: n  в ожидаемую сторону  вывод устойчив  \
гипотеза проверка                                               
Г2.4     3-я волна   224                 True           False   
         4-я волна   125                 True           False   

                    поправки сокращены  
гипотеза проверка                       
Г2.4     3-я волна               False  
         4-я волна               False

**Вывод. Г2.4 не подтверждается по правилу решения, но близко к порогу.** С ростом часов трудности совмещения как причину называют чаще: OR = 1,18 и 1,24 на 10 часов (p = 0,038 и 0,049 без поправки, 0,077 по Холму). Без сомнительных значений часов (вариант D) эффект сильнее и значим: OR = 1,34 и 1,58. В полных случаях — слабее и не значим. Описательно доля называющих эту причину растёт от нижней трети по часам к верхней: с 39% до 61% в 3-й волне и с 48% до 66% в 4-й (раздел З2.6).

### З2.5. Занятость и повторный выбор образования

In [19]:
for w in ['3', '4']:
    part = d[d[f'in_w{w}'] & d[f'fmt_{w}'].notna()]
    display(pd.crosstab(part[f'edu3_{w}'], part[f'fmt_{w}'], normalize='columns').round(3)
            .rename_axis(index=f'повторный выбор, {w}-я волна', columns='формат'))
    workers_w = part[part[f'emp_{w}'].eq(1)]
    tests = {'формат': assoc(part[f'fmt_{w}'].astype(str), part[f'edu3_{w}'].astype(str)),
             'терцили часов (работающие)': assoc(tertiles(workers_w[f'hours_{w}']).astype(str),
                                                workers_w[f'edu3_{w}'].astype(str))}
    if w == '4':
        tests['соответствие специальности'] = assoc(workers_w['fit_4'], workers_w['edu3_4'].astype(str))
    display(pd.DataFrame(tests).T.rename_axis(f'{w}-я волна'))
    smallest = part[f'edu3_{w}'].value_counts().min()
    print(f'{w}-я волна: самая малая категория повторного выбора — {smallest} человек; '
          f'на параметр мультиномиальной модели (11 параметров) — {smallest / 11:.1f} < 10, модель не строится.')
in4 = d[d['in_w4'] & d['edu3_4'].notna()]
pd.DataFrame({'все ответившие': in4['edu3_4'].value_counts(normalize=True),
              'без профилей A/B': in4.loc[~in4['tech_4w'], 'edu3_4'].value_counts(normalize=True)}).round(3)

формат,нет работы,полный день,неполный день,подработки
"повторный выбор, 3-я волна",,,,
"тот же вуз, та же специальность",0.499,0.429,0.510,0.457
"тот же вуз, другая специальность",0.149,0.224,0.130,0.196
"другой вуз, та же специальность",0.149,0.102,0.126,0.139
"другой вуз, другая специальность",0.139,0.184,0.170,0.170
не поступал бы,0.065,0.061,0.063,0.039


,n,p,V Крамера,метод
3-я волна,,,,
формат,1210,0.477126,0.056417,перестановки
терцили часов (работающие),454,0.535394,0.087874,χ²


3-я волна: самая малая категория повторного выбора — 72 человек; на параметр мультиномиальной модели (11 параметров) — 6.5 < 10, модель не строится.


формат,нет работы,полный день,неполный день,подработки
"повторный выбор, 4-я волна",,,,
"тот же вуз, та же специальность",0.501,0.512,0.452,0.456
"тот же вуз, другая специальность",0.149,0.214,0.199,0.181
"другой вуз, та же специальность",0.113,0.095,0.134,0.064
"другой вуз, другая специальность",0.189,0.143,0.177,0.269
не поступал бы,0.049,0.036,0.038,0.029


,n,p,V Крамера,метод
4-я волна,,,,
формат,912,0.172791,0.077527,перестановки
терцили часов (работающие),427,0.753062,0.077308,перестановки
соответствие специальности,427,0.00005,0.185895,перестановки


4-я волна: самая малая категория повторного выбора — 38 человек; на параметр мультиномиальной модели (11 параметров) — 3.5 < 10, модель не строится.


,все ответившие,без профилей A/B
edu3_4,,
"тот же вуз, та же специальность",0.299,0.483
"тот же вуз, другая специальность",0.105,0.169
"другой вуз, та же специальность",0.068,0.109
"другой вуз, другая специальность",0.503,0.198
не поступал бы,0.026,0.042


**Вывод.** Повторный выбор образования не связан с форматом занятости (p = 0,48 и 0,17) и с часами (p = 0,54 и 0,75). Связан с соответствием работы специальности в 4-й волне (V = 0,19, p < 0,001). Мультиномиальная модель не строилась: в самой малой категории («не поступал бы») меньше 10 наблюдений на параметр. Профили A/B сильно искажают распределение 4-й волны: с ними «другой вуз, другая специальность» выбрали 50%, без них — 20%. Поэтому распределения 4-й волны нужно приводить без профилей.

### З2.6. Причины мыслей об отчислении

In [20]:
REASON_NAMES = {f'reason{i}': name for i, name in REASONS.items()}
for w in ['3', '4']:
    base = d[d[f'reasons_cmp_{w}'] & d[f'emp_{w}'].notna()]
    table = pd.DataFrame({name: shares_by(base, f'{r}_{w}', f'fmt_{w}') for r, name in REASON_NAMES.items()}).T
    table['все'] = [share(base[f'{r}_{w}']) for r in REASON_NAMES]
    table['p (работа да/нет)'] = [assoc(base[f'emp_{w}'], base[f'{r}_{w}'])['p'] for r in REASON_NAMES]
    display(table.rename_axis(f'причина, {w}-я волна'))
    wk = base[base[f'emp_{w}'].eq(1)]
    display(pd.DataFrame({name: shares_by(wk.assign(t=tertiles(wk[f'hours_{w}'])), f'{r}_{w}', 't')
                          for r, name in REASON_NAMES.items()}).T.rename_axis(f'работающие, терцили часов, {w}-я волна'))
other = d[d['reasons_4w_any_st'].eq('VALID')]
print(f'4-я волна: «другое» отмечено — {share(other["reason_other_4"])}; '
      f'«затрудняюсь» без причины (статус DK) — {int(d["reasons_4w_any_st"].eq("DK").sum())}; '
      f'анкета прервана до блока — {int(d["reasons_4w_any_st"].eq("NOT_REACHED").sum())}. '
      'В 3-й волне вариантов «затрудняюсь» и «другое» не было.')

fmt_3,нет работы,полный день,неполный день,подработки,все,p (работа да/нет)
"причина, 3-я волна",,,,,,
разочарование в вузе,36% [30%; 43%] (80/221),29% [15%; 49%] (7/24),33% [25%; 43%] (37/111),33% [25%; 43%] (33/100),34% [30%; 39%] (157/456),4.406397e-01
разочарование в направлении,45% [39%; 52%] (100/221),50% [31%; 69%] (12/24),47% [38%; 56%] (52/111),48% [38%; 58%] (48/100),46% [42%; 51%] (212/456),6.059837e-01
академические трудности,57% [50%; 63%] (126/221),25% [12%; 45%] (6/24),43% [34%; 53%] (48/111),50% [40%; 60%] (50/100),50% [46%; 55%] (230/456),6.464541e-03
отношения с одногруппниками,7% [5%; 11%] (16/221),12% [4%; 31%] (3/24),8% [4%; 15%] (9/111),9% [5%; 16%] (9/100),8% [6%; 11%] (37/456),5.073238e-01
отношения с преподавателями,14% [10%; 19%] (30/221),17% [7%; 36%] (4/24),17% [11%; 25%] (19/111),12% [7%; 20%] (12/100),14% [11%; 18%] (65/456),6.872261e-01
экономические трудности семьи,14% [10%; 19%] (30/221),25% [12%; 45%] (6/24),17% [11%; 25%] (19/111),16% [10%; 24%] (16/100),16% [13%; 19%] (71/456),2.543956e-01
трудности совмещения с работой,10% [7%; 15%] (23/221),71% [51%; 85%] (17/24),50% [41%; 60%] (56/111),39% [30%; 49%] (39/100),30% [26%; 34%] (135/456),3.077466e-18


t,"(0.999, 10.0]","(10.0, 21.0]","(21.0, 168.0]"
"работающие, терцили часов, 3-я волна",,,
разочарование в вузе,34% [24%; 45%] (25/74),32% [22%; 44%] (20/63),39% [29%; 51%] (26/66)
разочарование в направлении,46% [35%; 57%] (34/74),54% [42%; 66%] (34/63),47% [35%; 59%] (31/66)
академические трудности,46% [35%; 57%] (34/74),43% [31%; 55%] (27/63),39% [29%; 51%] (26/66)
отношения с одногруппниками,7% [3%; 15%] (5/74),13% [7%; 23%] (8/63),8% [3%; 17%] (5/66)
отношения с преподавателями,12% [7%; 22%] (9/74),22% [14%; 34%] (14/63),17% [10%; 27%] (11/66)
экономические трудности семьи,15% [9%; 25%] (11/74),19% [11%; 30%] (12/63),18% [11%; 29%] (12/66)
трудности совмещения с работой,39% [29%; 51%] (29/74),49% [37%; 61%] (31/63),61% [49%; 71%] (40/66)


fmt_4,нет работы,полный день,неполный день,подработки,все,p (работа да/нет)
"причина, 4-я волна",,,,,,
разочарование в вузе,46% [39%; 54%] (77/166),21% [11%; 36%] (8/39),35% [24%; 47%] (23/66),38% [27%; 50%] (24/64),39% [34%; 45%] (132/335),9.538267e-03
разочарование в направлении,54% [47%; 62%] (90/166),31% [19%; 46%] (12/39),39% [29%; 51%] (26/66),61% [49%; 72%] (39/64),50% [45%; 55%] (167/335),1.131890e-01
академические трудности,49% [41%; 56%] (81/166),31% [19%; 46%] (12/39),38% [27%; 50%] (25/66),38% [27%; 50%] (24/64),42% [37%; 48%] (142/335),1.867803e-02
отношения с одногруппниками,8% [5%; 13%] (13/166),13% [6%; 27%] (5/39),9% [4%; 18%] (6/66),3% [1%; 11%] (2/64),8% [5%; 11%] (26/335),9.620772e-01
отношения с преподавателями,12% [8%; 18%] (20/166),28% [17%; 44%] (11/39),17% [10%; 27%] (11/66),16% [9%; 26%] (10/64),16% [12%; 20%] (52/335),8.179713e-02
экономические трудности семьи,13% [9%; 19%] (22/166),21% [11%; 36%] (8/39),11% [5%; 20%] (7/66),22% [14%; 33%] (14/64),15% [12%; 19%] (51/335),3.196602e-01
трудности совмещения с работой,18% [13%; 25%] (30/166),67% [51%; 79%] (26/39),47% [35%; 59%] (31/66),41% [29%; 53%] (26/64),34% [29%; 39%] (113/335),1.878612e-09


t,"(-0.001, 10.0]","(10.0, 22.667]","(22.667, 100.0]"
"работающие, терцили часов, 4-я волна",,,
разочарование в вузе,34% [24%; 47%] (21/61),40% [28%; 54%] (20/50),25% [16%; 38%] (14/56)
разочарование в направлении,51% [39%; 63%] (31/61),52% [39%; 65%] (26/50),36% [24%; 49%] (20/56)
академические трудности,39% [28%; 52%] (24/61),36% [24%; 50%] (18/50),32% [21%; 45%] (18/56)
отношения с одногруппниками,2% [0%; 9%] (1/61),14% [7%; 26%] (7/50),9% [4%; 19%] (5/56)
отношения с преподавателями,16% [9%; 28%] (10/61),20% [11%; 33%] (10/50),20% [11%; 32%] (11/56)
экономические трудности семьи,23% [14%; 35%] (14/61),18% [10%; 31%] (9/50),11% [5%; 21%] (6/56)
трудности совмещения с работой,48% [36%; 60%] (29/61),34% [22%; 48%] (17/50),66% [53%; 77%] (37/56)


4-я волна: «другое» отмечено — 7% [5%; 10%] (25/356); «затрудняюсь» без причины (статус DK) — 31; анкета прервана до блока — 329. В 3-й волне вариантов «затрудняюсь» и «другое» не было.


**Вывод.** Самые частые причины мыслей об отчислении — академические трудности (50% и 42%) и разочарование в направлении (46% и 50%). Трудности совмещения с работой называют 30% и 34%: у работающих полный день — 71% и 67%, на неполный день — 50% и 47%, на подработках — 39% и 41%. Академические трудности, наоборот, чаще у неработающих (57% против 25% у работающих полный день в 3-й волне). У части неработающих (10% и 18%) тоже названы трудности совмещения — видимо, они работали раньше: вопрос охватывает весь период обучения. «Другое» в 4-й волне отметили 7%; 31 человек ответил только «затрудняюсь».

## 10. ИВ3. Польза и трудности работы

Все проверки — среди работающих 4-й волны, порядковая логистическая регрессия, набор `work4`.

### Г3.1–Г3.3. Соответствие специальности, часы, польза и трудности

In [21]:
MOTIVE = "C(motive_g35_4, Treatment('содержание себя и оплата обучения'))"
work4_rows = lambda f: f['in_w4'] & f['emp_4'].eq(1)
rows = [test('Г3.1', '4-я волна', 'ordinal', 'benefit_4', ['fit_4', 'h10_4', MOTIVE], SOCIAL, work4_rows, 'work4', +1),
        test('Г3.2', '4-я волна', 'ordinal', 'harm_4', ['fit_4', 'h10_4', MOTIVE], SOCIAL, work4_rows, 'work4', -1),
        test('Г3.3', '4-я волна', 'ordinal', 'difficulty_4', ['h10_4', 'fit_4', MOTIVE], SOCIAL, work4_rows, 'work4', +1)]
ALL_ROWS += rows
show(report(rows))

,,модель,ожидаемый знак,A: n,эффект,A,B,C,D,p Холм (A),B: n,C: n,D: n,в ожидаемую сторону,вывод устойчив,OR по порогам (B),поправки сокращены
гипотеза,проверка,,,,,,,,,,,,,,,,
Г3.1,4-я волна,порядковая,+,376,OR,"2.28 [1.89; 2.75], p = 0.000","2.45 [1.90; 3.14], p = 0.000","2.36 [1.89; 2.95], p = 0.000","2.33 [1.89; 2.87], p = 0.000",0.000,220,261,306,True,True,≥2: 2.12; ≥3: 2.54; ≥4: 2.78,False
Г3.2,4-я волна,порядковая,−,380,OR,"0.94 [0.79; 1.12], p = 0.465","0.86 [0.68; 1.09], p = 0.222","0.91 [0.74; 1.12], p = 0.371","0.91 [0.75; 1.10], p = 0.320",0.465,224,265,312,True,True,≥2: 0.89; ≥3: 0.87; ≥4: 0.78,False
Г3.3,4-я волна,порядковая,+,382,OR,"1.20 [1.05; 1.37], p = 0.008","1.25 [1.05; 1.49], p = 0.011","1.22 [1.05; 1.41], p = 0.009","1.31 [1.09; 1.57], p = 0.004",0.008,224,267,312,True,True,≥2: 1.27; ≥3: 1.11; ≥4: 1.42,False


**Вывод.**
- **Г3.1 подтверждается.** Чем больше работа соответствует специальности, тем чаще её считают полезной для учёбы: OR = 2,28 [1,89; 2,75] на один пункт шкалы соответствия. Это самый сильный эффект в анализе; он устойчив во всех вариантах.
- **Г3.2 не подтверждается.** Соответствие специальности не связано с ощущением, что из-за работы страдает успеваемость (OR = 0,94, p = 0,47).
- **Г3.3 подтверждается.** Больше часов — больше трудностей совмещения: OR = 1,20 [1,05; 1,37] на 10 часов; устойчиво.

### Г3.4–Г3.5. Мотивы, польза и трудности

Г3.4: базовая категория — «содержание себя и оплата обучения»; Г3.5: базовая — «опыт и интерес». Проверка устойчивости — с перекодированными открытыми ответами.

In [22]:
M_SELF = "C(motive_g35_4, Treatment('содержание себя и оплата обучения'))"
M_EXP = "C(motive_g35_4, Treatment('опыт и интерес'))"
REC_SELF = "C(motive_g35_rec_4, Treatment('содержание себя и оплата обучения'))"
REC_EXP = "C(motive_g35_rec_4, Treatment('опыт и интерес'))"
rows = [
    test('Г3.4', '4-я волна: опыт и интерес', 'ordinal', 'benefit_4', [M_SELF, 'h10_4', 'fit_4'], SOCIAL, work4_rows,
         'work4', +1, focal=f'{M_SELF}[T.опыт и интерес]'),
    test('Г3.4', 'доп.: 4-я волна, доп. доход', 'ordinal', 'benefit_4', [M_SELF, 'h10_4', 'fit_4'], SOCIAL, work4_rows,
         'work4', +1, focal=f'{M_SELF}[T.доп. доход]'),
    test('Г3.4', 'устойч.: открытые ответы перекодированы', 'ordinal', 'benefit_4', [REC_SELF, 'h10_4', 'fit_4'], SOCIAL,
         work4_rows, 'work4', +1, focal=f'{REC_SELF}[T.опыт и интерес]'),
    test('Г3.5', '4-я волна: содержание себя и оплата обучения', 'ordinal', 'difficulty_4', [M_EXP, 'h10_4', 'fit_4'],
         SOCIAL, work4_rows, 'work4', +1, focal=f'{M_EXP}[T.содержание себя и оплата обучения]'),
    test('Г3.5', 'устойч.: открытые ответы перекодированы', 'ordinal', 'difficulty_4', [REC_EXP, 'h10_4', 'fit_4'],
         SOCIAL, work4_rows, 'work4', +1, focal=f'{REC_EXP}[T.содержание себя и оплата обучения]'),
]
ALL_ROWS += rows
show(report(rows))

модель  \
гипотеза проверка                                                   
Г3.4     4-я волна: опыт и интерес                     порядковая   
         доп.: 4-я волна, доп. доход                   порядковая   
         устойч.: открытые ответы перекодированы       порядковая   
Г3.5     4-я волна: содержание себя и оплата обучения  порядковая   
         устойч.: открытые ответы перекодированы       порядковая   

                                                      ожидаемый знак  A: n  \
гипотеза проверка                                                            
Г3.4     4-я волна: опыт и интерес                                 +   376   
         доп.: 4-я волна, доп. доход                               +   376   
         устойч.: открытые ответы перекодированы                   +   380   
Г3.5     4-я волна: содержание себя и оплата обучения              +   382   
         устойч.: открытые ответы перекодированы                   +   386   

                                                      эффект  \
гипотеза проверка                                              
Г3.4     4-я волна: опыт и интерес                        OR   
         доп.: 4-я волна, доп. доход                      OR   
         устойч.: открытые ответы перекодированы          OR   
Г3.5     4-я волна: содержание себя и оплата обучения     OR   
         устойч.: открытые ответы перекодированы          OR   

                                                                                  A  \
гипотеза проверка                                                                     
Г3.4     4-я волна: опыт и интерес                     1.58 [0.96; 2.61], p = 0.072   
         доп.: 4-я волна, доп. доход                   1.10 [0.68; 1.76], p = 0.705   
         устойч.: открытые ответы перекодированы       1.56 [0.95; 2.56], p = 0.081   
Г3.5     4-я волна: содержание себя и оплата обучения  2.52 [1.53; 4.15], p = 0.000   
         устойч.: открытые ответы перекодированы       2.44 [1.49; 4.01], p = 0.000   

                                                                                  B  \
гипотеза проверка                                                                     
Г3.4     4-я волна: опыт и интерес                     1.03 [0.54; 1.95], p = 0.933   
         доп.: 4-я волна, доп. доход                   1.02 [0.54; 1.92], p = 0.948   
         устойч.: открытые ответы перекодированы       1.02 [0.54; 1.93], p = 0.960   
Г3.5     4-я волна: содержание себя и оплата обучения  3.12 [1.61; 6.04], p = 0.001   
         устойч.: открытые ответы перекодированы       3.14 [1.62; 6.08], p = 0.001   

                                                                                  C  \
гипотеза проверка                                                                     
Г3.4     4-я волна: опыт и интерес                     1.38 [0.78; 2.45], p = 0.271   
         доп.: 4-я волна, доп. доход                   1.06 [0.60; 1.88], p = 0.850   
         устойч.: открытые ответы перекодированы       1.36 [0.77; 2.42], p = 0.288   
Г3.5     4-я волна: содержание себя и оплата обучения  2.55 [1.43; 4.55], p = 0.001   
         устойч.: открытые ответы перекодированы       2.57 [1.44; 4.58], p = 0.001   

                                                                                  D  \
гипотеза проверка                                                                     
Г3.4     4-я волна: опыт и интерес                     1.31 [0.76; 2.27], p = 0.334   
         доп.: 4-я волна, доп. доход                   1.19 [0.70; 2.04], p = 0.520   
         устойч.: открытые ответы перекодированы       1.27 [0.74; 2.20], p = 0.385   
Г3.5     4-я волна: содержание себя и оплата обучения  3.43 [1.95; 6.01], p = 0.000   
         устойч.: открытые ответы перекодированы       3.39 [1.93; 5.93], p = 0.000   

                                                       p Холм (A)  B: n  C: n  \
гипотеза проверка                                         

**Вывод.**
- **Г3.4 не подтверждается.** Работающие ради опыта и интереса чаще отмечают пользу, чем работающие ради содержания себя (OR = 1,58), но эффект не значим (p = 0,072, по Холму 0,14). В полных случаях эффекта почти нет (OR = 1,03). При этом польза уже объяснена соответствием специальности (Г3.1), которое в модели учтено.
- **Г3.5 подтверждается.** Работающие ради содержания себя и оплаты обучения испытывают больше трудностей совмещения, чем работающие ради опыта и интереса: OR = 2,52 [1,53; 4,15] при равных часах и соответствии специальности. Эффект устойчив во всех вариантах и при перекодировке открытых ответов.

### З3.6. Одновременная польза и трудности

«Согласен» — ответы 3–4 («скорее» и «полностью согласен»).

In [23]:
wk = d[d['emp_4'].eq(1) & d['benefit_difficulty_4'].notna()].assign(часы=lambda f: tertiles(f['hours_4']))
COMBOS = ['польза и трудности', 'только польза', 'только трудности', 'ни того, ни другого']
display(pd.Series({c: share(wk['benefit_difficulty_4'], c) for c in COMBOS}, name='работающие 4-й волны').to_frame())
for by in ['часы', 'fit_4', 'motive_g35_4', 'sphere_4']:
    table = pd.crosstab(wk[by], wk['benefit_difficulty_4'], normalize='index').reindex(columns=COMBOS).round(2)
    table['n'] = wk.groupby(by, observed=True).size()
    test_result = assoc(wk[by].astype(str), wk['benefit_difficulty_4'])
    display(table.rename_axis(f"{by}: p = {test_result['p']:.3f}, V = {test_result['V Крамера']:.2f}"))

,работающие 4-й волны
польза и трудности,18% [14%; 22%] (71/400)
только польза,40% [35%; 44%] (158/400)
только трудности,16% [13%; 20%] (65/400)
"ни того, ни другого",26% [22%; 31%] (106/400)


benefit_difficulty_4,польза и трудности,только польза,только трудности,"ни того, ни другого",n
"часы: p = 0.008, V = 0.15",,,,,
"(-0.001, 10.0]",0.14,0.36,0.21,0.29,146
"(10.0, 20.0]",0.13,0.51,0.12,0.24,124
"(20.0, 100.0]",0.26,0.34,0.15,0.25,122


benefit_difficulty_4,польза и трудности,только польза,только трудности,"ни того, ни другого",n
"fit_4: p = 0.000, V = 0.26",,,,,
1.0,0.09,0.24,0.27,0.39,127
2.0,0.18,0.23,0.19,0.40,62
3.0,0.25,0.42,0.14,0.20,101
4.0,0.20,0.66,0.05,0.09,101


benefit_difficulty_4,польза и трудности,только польза,только трудности,"ни того, ни другого",n
"motive_g35_4: p = 0.000, V = 0.21",,,,,
доп. доход,0.13,0.37,0.12,0.38,115
опыт и интерес,0.16,0.57,0.10,0.18,102
содержание себя и оплата обучения,0.24,0.31,0.22,0.23,165


benefit_difficulty_4,польза и трудности,только польза,только трудности,"ни того, ни другого",n
"sphere_4: p = 0.077, V = 0.17",,,,,
образование,0.23,0.39,0.12,0.26,66
наука,0.13,0.65,0.11,0.11,46
бизнес: наёмный работник,0.17,0.43,0.13,0.27,119
бизнес: предприниматель,0.07,0.50,0.14,0.29,14
госуправление,0.14,0.57,0.14,0.14,7
общественные и политические организации,0.36,0.18,0.27,0.18,11
СМИ,0.10,0.40,0.20,0.30,10
фриланс,0.19,0.30,0.20,0.31,74
другое,0.17,0.23,0.25,0.36,53


**Вывод.** Среди работающих 4-й волны пользу и трудности одновременно отмечают 18%, только пользу — 40%, только трудности — 16%, ни то, ни другое — 26%. Одновременные польза и трудности чаще при нагрузке больше 20 часов (26%); «только польза» растёт с соответствием специальности (с 24% до 66%). У работающих ради содержания себя и оплаты обучения чаще встречаются трудности (вместе с пользой — 24%, без неё — 22%); у работающих ради опыта и интереса — «только польза» (57%). По сферам различия не значимы (p = 0,08).

### З3.7. Мотивы работы и образовательные показатели

Порядковые модели оценок и удовлетворённости по мотиву (базовая категория — «доп. доход») с поправкой на часы и общий набор; для повторного выбора — χ².

In [24]:
MOTIVE_RAW = "C(motive_raw_{w}, Treatment('доп. доход'))"
rows = []
for w in ['3', '4']:
    term = MOTIVE_RAW.format(w=w)
    for y, label in [(f'mark_{w}', 'оценки'), (f'sat_{w}', 'удовлетворённость')]:
        for motive in ['содержать себя', 'опыт', 'интерес', 'оплата обучения']:
            rows.append(test('З3.7', f'{w}-я волна, {label}: {motive}', 'ordinal', y, [term, f'h10_{w}'], SOCIAL,
                             lambda f, w=w: f[f'in_w{w}'] & f[f'emp_{w}'].eq(1), f'work{w}', +1,
                             focal=f'{term}[T.{motive}]'))
z37 = show(report(rows)).drop(columns=['ожидаемый знак', 'в ожидаемую сторону', 'p Холм (A)'])
display(z37)
pd.DataFrame({f'{w}-я волна': assoc(d.loc[d[f'emp_{w}'].eq(1), f'motive_raw_{w}'].astype(str),
                                     d.loc[d[f'emp_{w}'].eq(1), f'edu3_{w}'].astype(str)) for w in ['3', '4']})

модель  A: n  \
гипотеза проверка                                                          
З3.7     3-я волна, оценки: содержать себя              порядковая   514   
         3-я волна, оценки: опыт                        порядковая   514   
         3-я волна, оценки: интерес                     порядковая   514   
         3-я волна, оценки: оплата обучения             порядковая   514   
         3-я волна, удовлетворённость: содержать себя   порядковая   505   
         3-я волна, удовлетворённость: опыт             порядковая   505   
         3-я волна, удовлетворённость: интерес          порядковая   505   
         3-я волна, удовлетворённость: оплата обучения  порядковая   505   
         4-я волна, оценки: содержать себя              порядковая   421   
         4-я волна, оценки: опыт                        порядковая   421   
         4-я волна, оценки: интерес                     порядковая   421   
         4-я волна, оценки: оплата обучения             порядковая   421   
         4-я волна, удовлетворённость: содержать себя   порядковая   400   
         4-я волна, удовлетворённость: опыт             порядковая   400   
         4-я волна, удовлетворённость: интерес          порядковая   400   
         4-я волна, удовлетворённость: оплата обучения  порядковая   400   

                                                       эффект  \
гипотеза проверка                                               
З3.7     3-я волна, оценки: содержать себя                 OR   
         3-я волна, оценки: опыт                           OR   
         3-я волна, оценки: интерес                        OR   
         3-я волна, оценки: оплата обучения                OR   
         3-я волна, удовлетворённость: содержать себя      OR   
         3-я волна, удовлетворённость: опыт                OR   
         3-я волна, удовлетворённость: интерес             OR   
         3-я волна, удовлетворённость: оплата обучения     OR   
         4-я волна, оценки: содержать себя                 OR   
         4-я волна, оценки: опыт                           OR   
         4-я волна, оценки: интерес                        OR   
         4-я волна, оценки: оплата обучения                OR   
         4-я волна, удовлетворённость: содержать себя      OR   
         4-я волна, удовлетворённость: опыт                OR   
         4-я волна, удовлетворённость: интерес             OR   
         4-я волна, удовлетворённость: оплата обучения     OR   

                                                                                    A  \
гипотеза проверка                                                                       
З3.7     3-я волна, оценки: содержать себя               0.64 [0.43; 0.95], p = 0.028   
         3-я волна, оценки: опыт                         1.19 [0.75; 1.91], p = 0.460   
         3-я волна, оценки: интерес                      0.80 [0.45; 1.42], p = 0.451   
         3-я волна, оценки: оплата обучения              1.02 [0.34; 3.05], p = 0.965   
         3-я волна, удовлетворённость: содержать себя    0.91 [0.59; 1.41], p = 0.679   
         3-я волна, удовлетворённость: опыт              1.35 [0.80; 2.27], p = 0.264   
         3-я волна, удовлетворённость: интерес           0.58 [0.31; 1.06], p = 0.077   
         3-я волна, удовлетворённость: оплата обучения   1.97 [0.56; 6.95], p = 0.293   
         4-я волна, оценки: содержать себя               0.77 [0.49; 1.21], p = 0.254   
         4-я волна, оценки: опыт                         1.90 [1.12; 3.23], p = 0.017   
         4-я волна, оценки: интерес                      1.00 [0.48; 2.09], p = 0.995   
         4-я волна, оценки: оплата обучения              0.81 [0.25; 2.63], p = 0.721   
         4-я волна, удовлетворённость: содержать себя    0.85 [0.50; 1.44], p = 0.542   
         4-я волна, удовлетворённость: опыт              1.24 [0.67; 2.28], p = 0.490   
         4-я волна, удовлетворённость: интерес           0.80 [0.34; 1.89], p = 0.614   
       

,3-я волна,4-я волна
n,532,440
p,0.083846,0.40843
V Крамера,0.117193,0.108443
метод,перестановки,перестановки


**Вывод (разведочно).** По сравнению с работающими ради доп. дохода:
- в 3-й волне у работающих ради содержания себя оценки ниже (OR = 0,64, p = 0,028);
- в 4-й волне у работающих ради опыта оценки выше (OR = 1,90, p = 0,017).

По удовлетворённости устойчивых различий нет. С повторным выбором образования мотив не связан (p = 0,08 и 0,41). Категории «оплата обучения» (12 человек) и «другое» в моделях есть, но их оценки неточны и не интерпретируются. Поправка на множественность для задач З не применялась.

### З3.8. Субъективный ущерб успеваемости и оценки

In [25]:
wk = d[d['emp_4'].eq(1) & d['harm_4'].notna() & d['mark_4'].notna()]
display(pd.crosstab(wk['harm_4'], wk['mark_4'], margins=True).rename_axis(index='ущерб (1–4)', columns='оценки (1–4)'))
rho = stats.spearmanr(wk['harm_4'], wk['mark_4']).statistic
boot = stats.bootstrap((wk['harm_4'].to_numpy(), wk['mark_4'].to_numpy()),
                       lambda x, y: stats.spearmanr(x, y).statistic, paired=True, n_resamples=5000, rng=rng)
print(f'Спирмен: ρ = {rho:.2f} [{boot.confidence_interval.low:.2f}; {boot.confidence_interval.high:.2f}], n = {len(wk)}')
show(report([test('З3.8', '4-я волна', 'ordinal', 'mark_4', ['harm_4', 'h10_4'], SOCIAL, work4_rows, 'work4', -1)]))

оценки (1–4),1.0,2.0,3.0,4.0,All
ущерб (1–4),,,,,
1.0,17,29,62,40,148
2.0,18,45,45,33,141
3.0,20,24,17,11,72
4.0,11,9,11,6,37
All,66,107,135,90,398


Спирмен: ρ = -0.21 [-0.30; -0.11], n = 398


,,модель,ожидаемый знак,A: n,эффект,A,B,C,D,p Холм (A),B: n,C: n,D: n,в ожидаемую сторону,вывод устойчив,OR по порогам (B),поправки сокращены
гипотеза,проверка,,,,,,,,,,,,,,,,
З3.8,4-я волна,порядковая,−,395,OR,"0.66 [0.54; 0.80], p = 0.000","0.62 [0.48; 0.81], p = 0.000","0.61 [0.48; 0.78], p = 0.000","0.67 [0.53; 0.83], p = 0.000",0.0,230,274,323,True,True,≥2: 0.56; ≥3: 0.56; ≥4: 0.82,False


**Вывод.** Субъективный ущерб согласуется с самоотчётными оценками: чем сильнее согласие с тем, что из-за работы страдает успеваемость, тем ниже оценки (ρ Спирмена = −0,21 [−0,30; −0,11]). При равных часах и социальных характеристиках каждый пункт шкалы ущерба снижает шансы более высокой категории оценок: OR = 0,66 [0,54; 0,80]. Эффект устойчив во всех вариантах.

## 11. ИВ4. Опыт стажировок

### З4.1. Распространённость и характеристики стажировок

In [26]:
display(pd.DataFrame({f'{w}-я волна': {'была стажировка': share(d.loc[d[f'in_w{w}'], f'intern_{w}']),
                                       'оплачивалась': share(d[f'intern_paid_{w}']),
                                       'соответствует направлению (3–4)': share(d[f'intern_fit_{w}'].ge(3).where(d[f'intern_fit_{w}'].notna()), True)}
                      for w in ['3', '4']}))
display(pd.DataFrame({f'{w}-я волна': d[f'intern_course_{w}'].value_counts(normalize=True).sort_index().round(3)
                      for w in ['3', '4']}).rename_axis('курс первой стажировки'))
display(pd.DataFrame({'мотив': d['intern_motive_raw_3'].value_counts(), 'доля': d['intern_motive_raw_3'].value_counts(normalize=True).round(3)}))
display(pd.DataFrame({v: mean_ci(d[v]) for v in ['intern_fit_3', 'intern_benefit_3', 'intern_harm_3', 'intern_difficulty_3', 'intern_fit_4']},
                     index=['среднее [95% ДИ] (1–4)']).T)
by_group = {}
for w in ['3', '4']:
    part = d[d[f'in_w{w}']]
    for g, label in [('sex_c', 'пол'), ('major_c', 'направление'), ('funding_c', 'финансирование'), (f'emp_{w}', 'есть работа')]:
        by_group[(f'{w}-я волна', label)] = {**assoc(part[g], part[f'intern_{w}']),
                                             'доли': '; '.join(f'{k}: {v}' for k, v in shares_by(part, f'intern_{w}', g).items())}
pd.DataFrame(by_group).T

,3-я волна,4-я волна
была стажировка,20% [18%; 22%] (241/1211),28% [25%; 31%] (258/913)
оплачивалась,59% [53%; 65%] (143/241),53% [47%; 59%] (136/258)
соответствует направлению (3–4),71% [65%; 77%] (165/232),82% [77%; 86%] (199/243)


,3-я волна,4-я волна
курс первой стажировки,,
1.0,0.212,0.124
2.0,0.485,0.252
3.0,0.303,0.516
4.0,NaN,0.109


,мотив,доля
intern_motive_raw_3,,
опыт,112,0.511
интерес,42,0.192
содержать себя,33,0.151
доп. доход,32,0.146
другое,0,0.000


,среднее [95% ДИ] (1–4)
intern_fit_3,"3.00 [2.86; 3.14], n=232"
intern_benefit_3,"2.74 [2.60; 2.87], n=219"
intern_harm_3,"1.89 [1.75; 2.02], n=223"
intern_difficulty_3,"2.06 [1.92; 2.19], n=218"
intern_fit_4,"3.19 [3.08; 3.31], n=243"


n         p V Крамера         метод  \
3-я волна пол             1109  0.064332  0.055549            χ²   
          направление     1123  0.014952  0.096566            χ²   
          финансирование   937  0.297502  0.050869            χ²   
          есть работа     1211       0.0  0.224692            χ²   
4-я волна пол              769  0.030357  0.078086            χ²   
          направление      751  0.000243  0.160106            χ²   
          финансирование   691  0.945003  0.013681  перестановки   
          есть работа      913       0.0  0.183356            χ²   

                                                                                                     доли  
3-я волна пол                         мужской: 22% [19%; 25%] (135/625); женский: 17% [14%; 21%] (83/484)  
          направление     инженерные: 19% [16%; 22%] (118/615); естественные: 20% [15%; 26%] (35/176);...  
          финансирование  бюджет: 17% [14%; 20%] (126/744); платное: 22% [16%; 29%] (34/153); целевое:...  
          есть работа                         0.0: 12% [10%; 15%] (81/678); 1.0: 30% [26%; 34%] (160/533)  
4-я волна пол                         мужской: 32% [27%; 36%] (130/409); женский: 25% [21%; 29%] (89/360)  
          направление     инженерные: 35% [30%; 39%] (139/402); естественные: 23% [16%; 31%] (25/111);...  
          финансирование  бюджет: 29% [25%; 33%] (156/539); платное: 27% [21%; 35%] (37/135); целевое:...  
          есть работа                         0.0: 20% [17%; 24%] (96/473); 1.0: 37% [32%; 41%] (162/440)

**Вывод.** Опыт стажировки есть у 20% участников 3-й волны и 28% — 4-й. Оплачивались 59% и 53% стажировок; соответствуют направлению обучения (ответы 3–4) 71% и 82%. Первую стажировку чаще проходили на 2-м курсе (по ответам 3-й волны) и на 3-м курсе (по ответам 4-й). Главный мотив — опыт (51%). Работающие студенты проходили стажировку в 2–2,5 раза чаще неработающих (30% против 12% и 37% против 20%, p < 0,001); доля различается по направлению (p = 0,015 и < 0,001) и слабо по полу (у мужчин чаще). Средние оценки стажировки: соответствие — 3,0 из 4, польза — 2,7, ущерб успеваемости — 1,9, трудности — 2,1.

### Г4.2–Г4.4. Соответствие стажировки, мотивы, польза и ущерб

Порядковая логистическая регрессия среди проходивших стажировку (3-я волна, набор `intern3`).

In [27]:
IM_BASE = "C(intern_motive_g44_3, Treatment('содержание себя и доп. доход'))"
intern_rows = lambda f: f['in_w3'] & f['intern_3'].eq(1)
rows = [test('Г4.2', '3-я волна', 'ordinal', 'intern_benefit_3', ['intern_fit_3', 'intern_paid_3', IM_BASE], SOCIAL,
             intern_rows, 'intern3', +1),
        test('Г4.3', '3-я волна', 'ordinal', 'intern_harm_3', ['intern_fit_3', 'intern_paid_3', IM_BASE], SOCIAL,
             intern_rows, 'intern3', -1),
        test('Г4.4', '3-я волна', 'ordinal', 'intern_benefit_3', [IM_BASE, 'intern_fit_3', 'intern_paid_3'], SOCIAL,
             intern_rows, 'intern3', +1, focal=f'{IM_BASE}[T.опыт и интерес]')]
ALL_ROWS += rows
show(report(rows))

,,модель,ожидаемый знак,A: n,эффект,A,B,C,D,p Холм (A),B: n,C: n,D: n,в ожидаемую сторону,вывод устойчив,OR по порогам (B),поправки сокращены
гипотеза,проверка,,,,,,,,,,,,,,,,
Г4.2,3-я волна,порядковая,+,202,OR,"2.94 [2.11; 4.11], p = 0.000","2.73 [1.78; 4.19], p = 0.000","3.11 [2.09; 4.63], p = 0.000","2.94 [2.11; 4.11], p = 0.000",0.000,118,136,202,True,True,≥2: 12.65; ≥3: 2.62; ≥4: 2.09,False
Г4.3,3-я волна,порядковая,−,204,OR,"0.58 [0.43; 0.78], p = 0.000","0.49 [0.34; 0.72], p = 0.000","0.53 [0.38; 0.74], p = 0.000","0.58 [0.43; 0.78], p = 0.000",0.000,116,136,204,True,True,≥2: 0.51; ≥3: 0.40; ≥4: 0.47,False
Г4.4,3-я волна,порядковая,+,202,OR,"2.13 [1.07; 4.22], p = 0.031","1.73 [0.69; 4.28], p = 0.240","1.79 [0.79; 4.08], p = 0.164","2.13 [1.07; 4.22], p = 0.031",0.031,118,136,202,True,False,≥2: 32.74; ≥3: 1.33; ≥4: 1.55,False


**Вывод.**
- **Г4.2 подтверждается.** Чем больше стажировка соответствует направлению, тем выше её польза для учёбы: OR = 2,94 [2,11; 4,11].
- **Г4.3 подтверждается.** Чем больше соответствие, тем меньше ущерб успеваемости: OR = 0,58 [0,43; 0,78].
- **Г4.4 подтверждается в основном варианте:** выбравшие стажировку ради опыта и интереса отмечают пользу чаще (OR = 2,13 [1,07; 4,22], p = 0,031). В полных случаях (118 человек) эффект того же направления, но не значим (OR = 1,73, p = 0,24) — вывод неустойчив.

Г4.2 и Г4.3 устойчивы во всех вариантах. На нижнем пороге шкалы мало ответов, поэтому отношение шансов там оценено неточно.

### З4.5. Оплачиваемые и неоплачиваемые стажировки

Для шкал — среднее с ДИ, критерий Манна — Уитни и вероятность превосходства P(оплачиваемая > неоплачиваемой).

In [28]:
rows = {}
for v, w in [('intern_fit_3', '3'), ('intern_benefit_3', '3'), ('intern_harm_3', '3'), ('intern_difficulty_3', '3'),
             ('intern_fit_4', '4')]:
    paid, unpaid = d.loc[d[f'intern_paid_{w}'].eq(1), v].dropna(), d.loc[d[f'intern_paid_{w}'].eq(0), v].dropna()
    u = stats.mannwhitneyu(paid, unpaid)
    rows[v] = {'оплачиваемые': mean_ci(paid), 'неоплачиваемые': mean_ci(unpaid), 'p (Манн — Уитни)': u.pvalue,
               'P(опл. > неопл.)': u.statistic / (len(paid) * len(unpaid))}
display(pd.DataFrame(rows).T)
motive = d[d['intern_motive_raw_3'].notna() & d['intern_paid_3'].notna()]
display(pd.crosstab(motive['intern_motive_raw_3'], motive['intern_paid_3'], normalize='columns').round(3)
        .rename(columns={0.0: 'неоплачиваемые', 1.0: 'оплачиваемые'}))
pd.Series(assoc(motive['intern_motive_raw_3'].astype(str), motive['intern_paid_3']), name='мотив × оплата').to_frame()

,оплачиваемые,неоплачиваемые,p (Манн — Уитни),P(опл. > неопл.)
intern_fit_3,"2.93 [2.75; 3.11], n=139","3.11 [2.89; 3.32], n=93",0.214745,0.454552
intern_benefit_3,"2.72 [2.54; 2.90], n=133","2.76 [2.54; 2.97], n=86",0.888069,0.494579
intern_harm_3,"1.96 [1.79; 2.14], n=135","1.77 [1.56; 1.98], n=88",0.139446,0.554714
intern_difficulty_3,"2.10 [1.92; 2.28], n=132","1.99 [1.77; 2.21], n=86",0.489311,0.526427
intern_fit_4,"3.12 [2.95; 3.30], n=129","3.27 [3.12; 3.43], n=114",0.395563,0.47076


intern_paid_3,неоплачиваемые,оплачиваемые
intern_motive_raw_3,,
содержать себя,0.082,0.194
доп. доход,0.082,0.187
опыт,0.647,0.425
интерес,0.188,0.194


,мотив × оплата
n,219
p,0.004269
V Крамера,0.245296
метод,χ²


**Вывод.** Оплачиваемые и неоплачиваемые стажировки одинаково оцениваются по соответствию, пользе, ущербу и трудностям (p = 0,14–0,89; вероятность превосходства 0,45–0,55 — практически равные шансы). Различаются мотивы (V = 0,25, p = 0,004): неоплачиваемые выбирают ради опыта (65% против 43%), оплачиваемые чаще — ради содержания себя и дохода (38% против 16%).

### З4.6. Сочетания текущей работы и опыта стажировки

Порядковые модели оценок и удовлетворённости по четырём сочетаниям (базовая — «нет работы, нет стажировки») с поправкой на общий набор; для повторного выбора — χ².

In [29]:
WI = "C(work_intern_{w}, Treatment('нет работы, нет стажировки'))"
rows = []
for w in ['3', '4']:
    term = WI.format(w=w)
    for y, label in [(f'mark_{w}', 'оценки'), (f'sat_{w}', 'удовлетворённость')]:
        for group in ['работа без стажировки', 'стажировка без работы', 'работа и стажировка']:
            rows.append(test('З4.6', f'{w}-я волна, {label}: {group}', 'ordinal', y, [term], SOCIAL,
                             lambda f, w=w: f[f'in_w{w}'], f'w{w}', +1, focal=f'{term}[T.{group}]'))
display(show(report(rows)).drop(columns=['ожидаемый знак', 'в ожидаемую сторону', 'p Холм (A)']))
display(pd.DataFrame({f'{w}-я волна': d[f'work_intern_{w}'].value_counts() for w in ['3', '4']}))
pd.DataFrame({f'{w}-я волна': assoc(d[f'work_intern_{w}'], d[f'edu3_{w}'].astype(str)) for w in ['3', '4']})

модель  \
гипотеза проверка                                                          
З4.6     3-я волна, оценки: работа без стажировки             порядковая   
         3-я волна, оценки: стажировка без работы             порядковая   
         3-я волна, оценки: работа и стажировка               порядковая   
         3-я волна, удовлетворённость: работа без стажировки  порядковая   
         3-я волна, удовлетворённость: стажировка без работы  порядковая   
         3-я волна, удовлетворённость: работа и стажировка    порядковая   
         4-я волна, оценки: работа без стажировки             порядковая   
         4-я волна, оценки: стажировка без работы             порядковая   
         4-я волна, оценки: работа и стажировка               порядковая   
         4-я волна, удовлетворённость: работа без стажировки  порядковая   
         4-я волна, удовлетворённость: стажировка без работы  порядковая   
         4-я волна, удовлетворённость: работа и стажировка    порядковая   

                                                              A: n эффект  \
гипотеза проверка                                                           
З4.6     3-я волна, оценки: работа без стажировки             1164     OR   
         3-я волна, оценки: стажировка без работы             1164     OR   
         3-я волна, оценки: работа и стажировка               1164     OR   
         3-я волна, удовлетворённость: работа без стажировки  1148     OR   
         3-я волна, удовлетворённость: стажировка без работы  1148     OR   
         3-я волна, удовлетворённость: работа и стажировка    1148     OR   
         4-я волна, оценки: работа без стажировки              861     OR   
         4-я волна, оценки: стажировка без работы              861     OR   
         4-я волна, оценки: работа и стажировка                861     OR   
         4-я волна, удовлетворённость: работа без стажировки   820     OR   
         4-я волна, удовлетворённость: стажировка без работы   820     OR   
         4-я волна, удовлетворённость: работа и стажировка     820     OR   

                                                                                         A  \
гипотеза проверка                                                                            
З4.6     3-я волна, оценки: работа без стажировки             0.96 [0.75; 1.22], p = 0.742   
         3-я волна, оценки: стажировка без работы             1.64 [1.05; 2.58], p = 0.031   
         3-я волна, оценки: работа и стажировка               1.42 [1.02; 1.96], p = 0.035   
         3-я волна, удовлетворённость: работа без стажировки  0.97 [0.74; 1.26], p = 0.804   
         3-я волна, удовлетворённость: стажировка без работы  1.06 [0.66; 1.72], p = 0.810   
         3-я волна, удовлетворённость: работа и стажировка    1.05 [0.73; 1.50], p = 0.808   
         4-я волна, оценки: работа без стажировки             0.76 [0.56; 1.02], p = 0.065   
         4-я волна, оценки: стажировка без работы             0.85 [0.57; 1.29], p = 0.456   
         4-я волна, оценки: работа и стажировка               0.99 [0.69; 1.43], p = 0.978   
         4-я волна, удовлетворённость: работа без стажировки  1.20 [0.86; 1.67], p = 0.286   
         4-я волна, удовлетворённость: стажировка без работы  1.20 [0.74; 1.95], p = 0.460   
         4-я волна, удовлетворённость: работа и стажировка    1.08 [0.73; 1.60], p = 0.699   

                                                                                         B  \
гипотеза проверка                                                                            
З4.6     3-я волна, оценки: работа без стажировки             0.95 [0.70; 1.28], p = 0.725   
         3-я волна, оценки: стажировка без работы             1.46 [0.80; 2.64], p = 0.214   
         3-я волна, оценки: работа и стажировка               1.59 [1.06; 2.39], p = 0.026   
         3-я волна, удовлетворённость: работа без стажировки  0.92 [0.66; 1.29], p = 0.642   
         3-я волна, удовлетворённость: ст

,3-я волна,4-я волна
"нет работы, нет стажировки",597,377
работа без стажировки,373,278
работа и стажировка,160,162
стажировка без работы,81,96


,3-я волна,4-я волна
n,1210,911
p,0.211039,0.510074
V Крамера,0.065403,0.064238
метод,перестановки,перестановки


**Вывод (разведочно).** В 3-й волне у студентов с опытом стажировки оценки выше, чем у тех, кто не работает и не стажировался: OR = 1,64 для стажировки без работы и 1,42 для работы со стажировкой (p ≈ 0,03). Работа без стажировки с оценками не связана (OR = 0,96). В 4-й волне различий по оценкам нет. По удовлетворённости и повторному выбору образования различий нет ни в одной волне. Порядок событий неизвестен: стажировка могла быть до текущей работы.

## 12. ИВ5. Занятость и здоровье

### Г5.1–Г5.2. Рабочие часы и сон

Линейная регрессия с устойчивыми ошибками HC3 среди работающих. Коэффициент — изменение сна (в часах) на 10 рабочих часов в неделю.

In [30]:
rows = []
for w in ['3', '4']:
    rows.append(test('Г5.1', f'{w}-я волна', 'ols', f'sleep_dur_wd_{w}', [f'h10_{w}'], SOCIAL,
                     lambda f, w=w: f[f'in_w{w}'] & f[f'emp_{w}'].eq(1), f'work{w}', -1))
for w in ['3', '4']:
    rows.append(test('Г5.2', f'{w}-я волна', 'ols', f'sjl_{w}', [f'h10_{w}'], SOCIAL,
                     lambda f, w=w: f[f'in_w{w}'] & f[f'emp_{w}'].eq(1), f'work{w}', +1))
ALL_ROWS += rows
show(report(rows))

модель ожидаемый знак  A: n эффект  \
гипотеза проверка                                          
Г5.1     3-я волна  МНК, HC3              −   528      b   
         4-я волна  МНК, HC3              −   433      b   
Г5.2     3-я волна  МНК, HC3              +   527      b   
         4-я волна  МНК, HC3              +   431      b   

                                                  A  \
гипотеза проверка                                     
Г5.1     3-я волна  -0.15 [-0.28; -0.02], p = 0.022   
         4-я волна   -0.05 [-0.17; 0.07], p = 0.408   
Г5.2     3-я волна  -0.06 [-0.12; -0.00], p = 0.041   
         4-я волна    0.04 [-0.06; 0.14], p = 0.410   

                                                 B  \
гипотеза проверка                                    
Г5.1     3-я волна  -0.19 [-0.39; 0.02], p = 0.075   
         4-я волна  -0.02 [-0.21; 0.17], p = 0.838   
Г5.2     3-я волна  -0.08 [-0.16; 0.00], p = 0.050   
         4-я волна  -0.03 [-0.11; 0.05], p = 0.508   

                                                 C  \
гипотеза проверка                                    
Г5.1     3-я волна  -0.16 [-0.35; 0.03], p = 0.101   
         4-я волна  -0.01 [-0.16; 0.15], p = 0.930   
Г5.2     3-я волна  -0.06 [-0.14; 0.01], p = 0.092   
         4-я волна  -0.01 [-0.09; 0.06], p = 0.727   

                                                 D  p Холм (A)  B: n  C: n  \
гипотеза проверка                                                            
Г5.1     3-я волна  -0.08 [-0.20; 0.04], p = 0.171       0.044   285   334   
         4-я волна   0.06 [-0.06; 0.18], p = 0.346       0.408   247   295   
Г5.2     3-я волна  -0.06 [-0.13; 0.02], p = 0.140       0.082   285   334   
         4-я волна  -0.01 [-0.10; 0.08], p = 0.829       0.410   246   294   

                    D: n  в ожидаемую сторону  вывод устойчив  \
гипотеза проверка                                               
Г5.1     3-я волна   476                 True           False   
         4-я волна   341                 True            True   
Г5.2     3-я волна   463                False            True   
         4-я волна   328                 True            True   

                    поправки сокращены  
гипотеза проверка                       
Г5.1     3-я волна               False  
         4-я волна               False  
Г5.2     3-я волна               False  
         4-я волна               False

**Вывод.**
- **Г5.1 частично подтверждается.** В 3-й волне на каждые 10 рабочих часов приходится на 0,15 часа (около 9 минут) меньше ночного интервала сна в будни: [−0,28; −0,02], p по Холму = 0,044. Но эффект неустойчив: в полных случаях p = 0,075, без сомнительных значений сна и часов (вариант D) — b = −0,08, p = 0,17. В 4-й волне связи нет (b = −0,05, p = 0,41).
- **Г5.2 не подтверждается.** В 3-й волне сдвиг середины сна с ростом часов даже немного уменьшается (b = −0,06, p = 0,041 — противоположно гипотезе); в 4-й связи нет.

### Г5.3. Работа полный день и занятия спортом

Основная модель — полный день против без работы; дополнительные контрасты — другие форматы.

In [31]:
rows = []
for w in ['3', '4']:
    rows.append(test('Г5.3', f'{w}-я волна', 'logit', f'pe_{w}', [f'full_{w}'], SOCIAL,
                     lambda f, w=w: f[f'in_w{w}'] & f[f'fmt_{w}'].isin(['полный день', 'нет работы']), f'w{w}', -1))
    for focal, name in [(f'part_{w}', 'неполный день'), (f'odd_{w}', 'подработки')]:
        rows.append(test('Г5.3', f'доп.: {w}-я волна, {name} против без работы', 'logit', f'pe_{w}',
                         [f'full_{w}', f'part_{w}', f'odd_{w}'], SOCIAL, lambda f, w=w: f[f'in_w{w}'], f'w{w}', -1,
                         focal=focal))
ALL_ROWS += rows
display(show(report(rows)))
pd.DataFrame({f'{w}-я волна': shares_by(d[d[f'in_w{w}']], f'pe_{w}', f'fmt_{w}') for w in ['3', '4']})

модель  \
гипотеза проверка                                                  
Г5.3     3-я волна                                         логит   
         доп.: 3-я волна, неполный день против без работы  логит   
         доп.: 3-я волна, подработки против без работы     логит   
         4-я волна                                         логит   
         доп.: 4-я волна, неполный день против без работы  логит   
         доп.: 4-я волна, подработки против без работы     логит   

                                                          ожидаемый знак  \
гипотеза проверка                                                          
Г5.3     3-я волна                                                     −   
         доп.: 3-я волна, неполный день против без работы              −   
         доп.: 3-я волна, подработки против без работы                 −   
         4-я волна                                                     −   
         доп.: 4-я волна, неполный день против без работы              −   
         доп.: 4-я волна, подработки против без работы                 −   

                                                           A: n эффект  \
гипотеза проверка                                                        
Г5.3     3-я волна                                          720     OR   
         доп.: 3-я волна, неполный день против без работы  1201     OR   
         доп.: 3-я волна, подработки против без работы     1201     OR   
         4-я волна                                          548     OR   
         доп.: 4-я волна, неполный день против без работы   901     OR   
         доп.: 4-я волна, подработки против без работы      901     OR   

                                                                                      A  \
гипотеза проверка                                                                         
Г5.3     3-я волна                                         0.98 [0.52; 1.88], p = 0.963   
         доп.: 3-я волна, неполный день против без работы  1.28 [0.92; 1.78], p = 0.148   
         доп.: 3-я волна, подработки против без работы     1.32 [0.93; 1.87], p = 0.120   
         4-я волна                                         1.75 [1.06; 2.88], p = 0.028   
         доп.: 4-я волна, неполный день против без работы  1.89 [1.31; 2.73], p = 0.001   
         доп.: 4-я волна, подработки против без работы     1.31 [0.91; 1.89], p = 0.148   

                                                                                      B  \
гипотеза проверка                                                                         
Г5.3     3-я волна                                         0.98 [0.44; 2.18], p = 0.964   
         доп.: 3-я волна, неполный день против без работы  1.13 [0.76; 1.67], p = 0.551   
         доп.: 3-я волна, подработки против без работы     1.51 [0.98; 2.34], p = 0.063   
         4-я волна                                         1.75 [0.96; 3.18], p = 0.069   
         доп.: 4-я волна, неполный день против без работы  1.92 [1.19; 3.11], p = 0.008   
         доп.: 4-я волна, подработки против без работы     1.22 [0.77; 1.93], p = 0.405   

                                                                                      C  \
гипотеза проверка                                                                         
Г5.3     3-я волна                                         0.91 [0.43; 1.94], p = 0.812   
         доп.: 3-я волна, неполный день против без работы  1.21 [0.83; 1.77], p = 0.314   
         доп.: 3-я волна, подработки против без работы     1.31 [0.88; 1.94], p = 0.182   
         4-я волна                                         1.64 [0.94; 2.86], p = 0.079   
         доп.: 4-я волна, неполный день против без работы  1.95 [1.25; 3.04], p = 0.003   
         доп.: 4-я волна, подработки против без работы     1.20 [0.79; 1.82], p = 0.398   

                                                                                      D  \
гипотеза проверка                

,3-я волна,4-я волна
нет работы,70% [66%; 73%] (470/676),54% [49%; 58%] (253/471)
полный день,69% [55%; 80%] (34/49),67% [56%; 76%] (56/84)
неполный день,74% [69%; 79%] (188/253),69% [62%; 76%] (129/186)
подработки,75% [69%; 80%] (173/230),61% [54%; 68%] (104/170)


**Вывод. Г5.3 не подтверждается.** В 3-й волне работающие полный день занимаются спортом так же часто, как неработающие (69% и 70%, OR = 0,98). В 4-й волне — наоборот, чаще (67% против 54%, OR = 1,75): это противоположно гипотезе. Дополнительно: в 4-й волне работающие неполный день тоже занимаются спортом чаще неработающих (69%, OR = 1,89, p по Холму = 0,004). Работа не вытесняет спорт — скорее, работающие и спортивные студенты пересекаются.

### Г5.4. Рабочие часы и частота спорта (3-я волна)

Порядковая регрессия частоты (1 — раз в месяц или реже … 5 — каждый день) среди работающих, занимающихся спортом.

In [32]:
rows = [test('Г5.4', '3-я волна', 'ordinal', 'pe_freq_3', ['h10_3'], SOCIAL,
             lambda f: f['in_w3'] & f['emp_3'].eq(1) & f['pe_3'].eq(1), 'work3', -1)]
ALL_ROWS += rows
show(report(rows))

,,модель,ожидаемый знак,A: n,эффект,A,B,C,D,p Холм (A),B: n,C: n,D: n,в ожидаемую сторону,вывод устойчив,OR по порогам (B),поправки сокращены
гипотеза,проверка,,,,,,,,,,,,,,,,
Г5.4,3-я волна,порядковая,−,393,OR,"0.94 [0.81; 1.08], p = 0.388","0.99 [0.82; 1.21], p = 0.946","0.98 [0.82; 1.17], p = 0.843","0.94 [0.79; 1.10], p = 0.434",0.388,208,247,375,True,True,≥2: 0.98; ≥3: 0.97; ≥4: 0.95; ≥5: 1.20,False


**Вывод. Г5.4 не подтверждается.** Среди работающих, занимающихся спортом, часы не связаны с частотой занятий: OR = 0,94 на 10 часов, p = 0,39. Вывод устойчив.

### З5.5. Курение и алкоголь

Только содержательные ответы «да»/«нет»; модели — логистическая регрессия на формат (базовая — без работы) с поправкой на общий набор.

In [33]:
rows = []
for w in ['3', '4']:
    for y, label in [(f'smoke_{w}', 'курение'), (f'alco_{w}', 'алкоголь')]:
        for focal, name in [(f'full_{w}', 'полный день'), (f'part_{w}', 'неполный день'), (f'odd_{w}', 'подработки')]:
            rows.append(test('З5.5', f'{w}-я волна, {label}: {name}', 'logit', y, [f'full_{w}', f'part_{w}', f'odd_{w}'],
                             SOCIAL, lambda f, w=w: f[f'in_w{w}'], f'w{w}', +1, focal=focal))
display(show(report(rows)).drop(columns=['ожидаемый знак', 'в ожидаемую сторону', 'p Холм (A)']))
for w in ['3', '4']:
    part = d[d[f'in_w{w}']]
    display(pd.DataFrame({'курение': shares_by(part, f'smoke_{w}', f'fmt_{w}'),
                          'алкоголь': shares_by(part, f'alco_{w}', f'fmt_{w}')}).rename_axis(f'формат, {w}-я волна'))
    wk = part[part[f'emp_{w}'].eq(1)].assign(часы=lambda f: tertiles(f[f'hours_{w}']))
    display(pd.DataFrame({'курение': shares_by(wk, f'smoke_{w}', 'часы'), 'алкоголь': shares_by(wk, f'alco_{w}', 'часы')})
            .rename_axis(f'работающие, терцили часов, {w}-я волна'))
pd.DataFrame({f'{w}-я волна, {v}': d.loc[d[f'in_w{w}'], f'{v}_{w}w_st'].value_counts()
              for w in ['3', '4'] for v in ['smoking', 'alco']}).fillna(0).astype(int).rename_axis('статус ответа')

модель  A: n эффект  \
гипотеза проверка                                                 
З5.5     3-я волна, курение: полный день     логит  1094     OR   
         3-я волна, курение: неполный день   логит  1094     OR   
         3-я волна, курение: подработки      логит  1094     OR   
         3-я волна, алкоголь: полный день    логит  1074     OR   
         3-я волна, алкоголь: неполный день  логит  1074     OR   
         3-я волна, алкоголь: подработки     логит  1074     OR   
         4-я волна, курение: полный день     логит   824     OR   
         4-я волна, курение: неполный день   логит   824     OR   
         4-я волна, курение: подработки      логит   824     OR   
         4-я волна, алкоголь: полный день    логит   811     OR   
         4-я волна, алкоголь: неполный день  логит   811     OR   
         4-я волна, алкоголь: подработки     логит   811     OR   

                                                                        A  \
гипотеза проверка                                                           
З5.5     3-я волна, курение: полный день     2.35 [1.22; 4.54], p = 0.011   
         3-я волна, курение: неполный день   2.26 [1.60; 3.19], p = 0.000   
         3-я волна, курение: подработки      1.49 [1.02; 2.18], p = 0.039   
         3-я волна, алкоголь: полный день    1.66 [0.84; 3.27], p = 0.142   
         3-я волна, алкоголь: неполный день  1.66 [1.21; 2.27], p = 0.002   
         3-я волна, алкоголь: подработки     1.25 [0.90; 1.73], p = 0.179   
         4-я волна, курение: полный день     2.02 [1.16; 3.53], p = 0.013   
         4-я волна, курение: неполный день   1.32 [0.86; 2.04], p = 0.206   
         4-я волна, курение: подработки      1.73 [1.12; 2.66], p = 0.014   
         4-я волна, алкоголь: полный день    1.66 [0.98; 2.78], p = 0.057   
         4-я волна, алкоголь: неполный день  1.24 [0.86; 1.81], p = 0.254   
         4-я волна, алкоголь: подработки     1.63 [1.11; 2.41], p = 0.014   

                                                                        B  \
гипотеза проверка                                                           
З5.5     3-я волна, курение: полный день     2.61 [1.19; 5.74], p = 0.017   
         3-я волна, курение: неполный день   1.99 [1.28; 3.07], p = 0.002   
         3-я волна, курение: подработки      0.98 [0.59; 1.60], p = 0.924   
         3-я волна, алкоголь: полный день    1.06 [0.44; 2.51], p = 0.902   
         3-я волна, алкоголь: неполный день  1.33 [0.90; 1.96], p = 0.155   
         3-я волна, алкоголь: подработки     1.05 [0.70; 1.57], p = 0.829   
         4-я волна, курение: полный день     2.12 [1.07; 4.19], p = 0.031   
         4-я волна, курение: неполный день   1.28 [0.71; 2.31], p = 0.413   
         4-я волна, курение: подработки      1.97 [1.14; 3.39], p = 0.015   
         4-я волна, алкоголь: полный день    1.79 [0.93; 3.46], p = 0.083   
         4-я волна, алкоголь: неполный день  1.22 [0.74; 1.99], p = 0.434   
         4-я волна, алкоголь: подработки     1.62 [0.99; 2.65], p = 0.054   

                                                                        C  \
гипотеза проверка                                                           
З5.5     3-я волна, курение: полный день     2.96 [1.38; 6.32], p = 0.005   
         3-я волна, курение: неполный день   2.22 [1.48; 3.34], p = 0.000   
         3-я волна, курение: подработки      1.26 [0.81; 1.97], p = 0.297   
         3-я волна, алкоголь: полный день    1.22 [0.54; 2.77], p = 0.627   
         3-я волна, алкоголь: неполный день  1.50 [1.04; 2.16], p = 0.031   
         3-я волна, алкоголь: подработки     0.99 [0.68; 1.42], p = 0.936   
         4-я волна, курение: полный день     1.88 [0.99; 3.58], p = 0.052   
         4-я волна, курение: неполный день   1.38 [0.81; 2.35], p = 0.238   
         4-я волна, курение: подработки      1.90 [1.15; 3.14], p = 0.012   
         4-я волна, алкоголь: полный день    1.50 [0.84; 2.69], p = 0.171   
         4-я волна, алк

,курение,алкоголь
"формат, 3-я волна",,
нет работы,19% [16%; 23%] (121/626),49% [45%; 53%] (299/611)
полный день,36% [23%; 50%] (16/45),61% [45%; 74%] (23/38)
неполный день,34% [28%; 41%] (78/227),61% [55%; 67%] (141/230)
подработки,26% [21%; 33%] (53/203),54% [47%; 61%] (109/202)


,курение,алкоголь
"работающие, терцили часов, 3-я волна",,
"(0.999, 10.0]",28% [21%; 35%] (42/152),52% [44%; 59%] (81/157)
"(10.0, 20.0]",26% [19%; 34%] (33/128),59% [51%; 68%] (73/123)
"(20.0, 168.0]",44% [36%; 53%] (55/124),64% [56%; 72%] (78/121)


,курение,алкоголь
"формат, 4-я волна",,
нет работы,19% [16%; 23%] (82/433),53% [48%; 57%] (223/423)
полный день,32% [23%; 44%] (24/74),65% [54%; 75%] (49/75)
неполный день,24% [18%; 31%] (41/172),59% [52%; 67%] (98/165)
подработки,29% [22%; 37%] (45/155),66% [58%; 73%] (104/158)


,курение,алкоголь
"работающие, терцили часов, 4-я волна",,
"(-0.001, 10.0]",31% [24%; 39%] (46/147),66% [57%; 73%] (95/145)
"(10.0, 20.0]",22% [15%; 30%] (27/124),62% [52%; 70%] (72/117)
"(20.0, 168.0]",27% [20%; 36%] (33/122),63% [54%; 71%] (79/126)


,"3-я волна, smoking","3-я волна, alco","4-я волна, smoking","4-я волна, alco"
статус ответа,,,,
MISS,45,0,0,0
NOT_REACHED,78,143,602,602
REF,0,0,77,90
VALID,1101,1081,834,821


**Вывод (разведочно).** Работающие курят чаще неработающих: 26–36% против 19%. В 3-й волне OR = 2,35 (полный день), 2,26 (неполный), 1,49 (подработки); в 4-й — 2,02 (полный день) и 1,73 (подработки). Алкоголь тоже чаще у работающих (54–66% против 49–53%), но значимо не во всех группах: неполный день в 3-й волне (OR = 1,66) и подработки в 4-й (OR = 1,63). Связь с часами неоднозначна. В 4-й волне отказались отвечать о курении 77 человек, об алкоголе — 90; в 3-й волне явных отказов нет, а 45 пустых ответов о курении оставлены пропусками неизвестного происхождения.

### З5.6. Режим сна по форматам занятости

Время — в часах от полуночи предыдущего дня (24 = 0:00, 25 = 1:00, 31 = 7:00).

In [34]:
for w, (bed_wd, bed_we, wake_wd, wake_we) in SLEEP_ITEMS.items():
    part = d[d[f'in_w{w}'] & d[f'fmt_{w}'].notna()].assign(
        **{'отбой, будни': 19 + d[bed_wd], 'подъём, будни': 26 + d[wake_wd],
           'отбой, выходные': 19 + d[bed_we], 'подъём, выходные': 26 + d[wake_we]})
    cols = ['отбой, будни', 'подъём, будни', 'отбой, выходные', 'подъём, выходные',
            f'sleep_dur_wd_{w}', f'sleep_dur_we_{w}', f'sjl_{w}']
    table = pd.DataFrame({c: part.groupby(f'fmt_{w}', observed=True)[c].apply(median_iqr) for c in cols})
    table.loc['p (Краскел — Уоллис)'] = [stats.kruskal(*[g.dropna() for _, g in part.groupby(f'fmt_{w}', observed=True)[c]]).pvalue
                                         for c in cols]
    display(table.T.rename_axis(f'{w}-я волна: медиана (IQR)'))
    print(f'{w}-я волна: цензурированы (отбой 6:00 или подъём 14:00) — будни {share(d.loc[d[f"in_w{w}"], f"sleep_wd_{w}_censored"].where(d[f"sleep_dur_wd_{w}"].notna()), True)}, '
          f'выходные {share(d.loc[d[f"in_w{w}"], f"sleep_we_{w}_censored"].where(d[f"sleep_dur_we_{w}"].notna()), True)}')

fmt_3,нет работы,полный день,неполный день,подработки,p (Краскел — Уоллис)
3-я волна: медиана (IQR),,,,,
"отбой, будни","24.0 (24.0–26.0), n=677","25.0 (24.0–26.0), n=49","25.0 (24.0–26.0), n=253","25.0 (24.0–26.0), n=230",0.246538
"подъём, будни","32.0 (31.0–33.0), n=676","32.0 (31.0–33.0), n=49","32.0 (31.0–33.0), n=253","32.0 (31.0–33.0), n=230",0.164871
"отбой, выходные","25.0 (24.0–26.0), n=677","25.0 (24.0–27.0), n=49","25.0 (24.0–27.0), n=253","25.0 (24.0–27.0), n=230",0.436615
"подъём, выходные","34.0 (33.0–36.0), n=676","34.0 (32.0–35.0), n=49","34.0 (33.0–36.0), n=253","34.0 (33.0–36.0), n=230",0.365911
sleep_dur_wd_3,"8.0 (7.0–9.0), n=670","7.0 (5.0–8.0), n=48","8.0 (6.0–8.0), n=252","7.0 (6.0–8.0), n=230",0.000349
sleep_dur_we_3,"9.0 (8.0–10.0), n=672","9.0 (8.0–10.0), n=48","9.0 (8.0–10.0), n=251","9.0 (8.0–10.0), n=230",0.050471
sjl_3,"1.5 (0.5–2.0), n=669","1.5 (0.5–2.1), n=48","1.5 (0.5–2.0), n=251","1.5 (0.5–2.0), n=230",0.945294


3-я волна: цензурированы (отбой 6:00 или подъём 14:00) — будни 3% [2%; 4%] (32/1200), выходные 8% [7%; 10%] (96/1201)


fmt_4,нет работы,полный день,неполный день,подработки,p (Краскел — Уоллис)
4-я волна: медиана (IQR),,,,,
"отбой, будни","24.0 (23.0–26.0), n=471","24.0 (23.0–26.0), n=84","24.0 (23.0–25.0), n=186","25.0 (24.0–26.0), n=170",0.00051
"подъём, будни","32.0 (31.0–33.0), n=471","32.0 (31.0–33.0), n=84","32.0 (31.0–33.0), n=186","32.0 (31.0–33.0), n=170",0.000126
"отбой, выходные","25.0 (24.0–26.0), n=471","25.0 (24.0–26.2), n=84","25.0 (24.0–26.0), n=186","25.0 (24.0–26.0), n=170",0.118991
"подъём, выходные","34.0 (33.0–36.0), n=471","34.0 (32.0–36.0), n=84","34.0 (33.0–35.0), n=186","34.0 (33.0–35.0), n=170",0.240556
sleep_dur_wd_4,"8.0 (7.0–9.0), n=468","8.0 (6.5–9.0), n=83","8.0 (7.0–9.0), n=185","8.0 (7.0–9.0), n=168",0.144577
sleep_dur_we_4,"9.0 (8.0–10.0), n=467","9.0 (8.0–10.0), n=81","9.0 (8.0–10.0), n=185","9.0 (8.0–10.0), n=168",0.859174
sjl_4,"1.0 (0.5–2.0), n=467","1.0 (0.5–2.5), n=81","1.5 (0.5–2.0), n=185","1.0 (0.5–2.0), n=168",0.452073


4-я волна: цензурированы (отбой 6:00 или подъём 14:00) — будни 3% [2%; 4%] (25/904), выходные 8% [6%; 10%] (69/901)


**Вывод.** Режим сна по форматам различается мало. Типичный отбой — около 0:00–1:00, подъём в будни — около 8:00. В 3-й волне ночной интервал в будни у работающих полный день и на подработках короче (медиана 7 часов против 8 у неработающих, p < 0,001). В 4-й волне различается время отбоя и подъёма (p < 0,001), но не длительность сна. Сдвиг середины сна от формата не зависит. Цензурированы (крайние варианты шкалы) 3% ответов о буднях и 8% — о выходных.

## 13. ИВ6. Планы распределения времени и трудоустройства

### З6.1. Планируемое распределение времени (3-я волна)

In [35]:
alloc = d[d['hours_3w_1_st'].eq('VALID')]
summary = pd.DataFrame({'доля нулей': (alloc[ALLOC_COLS] == 0).mean().round(2),
                        'медиана среди > 0': alloc[ALLOC_COLS].where(alloc[ALLOC_COLS] > 0).median().round(2),
                        'IQR среди > 0': [f"{alloc[c][alloc[c] > 0].quantile(.25):.2f}–{alloc[c][alloc[c] > 0].quantile(.75):.2f}"
                                          for c in ALLOC_COLS]})
display(summary.rename_axis(f'все, n = {len(alloc)}'))
for by in ['fmt_3', 'motive_raw_3']:
    group = alloc[alloc[by].notna()]
    display(pd.concat({'доля нулей': group.groupby(by, observed=True)[ALLOC_COLS].apply(lambda x: (x == 0).mean()),
                       'медиана среди > 0': group.groupby(by, observed=True)[ALLOC_COLS].apply(lambda x: x.where(x > 0).median())},
                      axis=1).round(2).assign(n=group.groupby(by, observed=True).size()))

,доля нулей,медиана среди > 0,IQR среди > 0
"все, n = 1220",,,
alloc_учёба_3,0.02,0.50,0.40–0.70
alloc_клубы_3,0.71,0.11,0.05–0.22
alloc_работа_3,0.44,0.26,0.17–0.40
alloc_волонтёрство_3,0.89,0.08,0.03–0.11
alloc_курсы_3,0.63,0.12,0.08–0.20
alloc_остальное_3,0.26,0.20,0.10–0.32


доля нулей                                                    \
              alloc_учёба_3 alloc_клубы_3 alloc_работа_3 alloc_волонтёрство_3   
fmt_3                                                                           
нет работы             0.01          0.73           0.67                 0.91   
полный день            0.04          0.67           0.14                 0.88   
неполный день          0.03          0.70           0.09                 0.88   
подработки             0.05          0.69           0.19                 0.84   

                                              медиана среди > 0                \
              alloc_курсы_3 alloc_остальное_3     alloc_учёба_3 alloc_клубы_3   
fmt_3                                                                           
нет работы             0.65              0.21              0.60          0.12   
полный день            0.63              0.57              0.48          0.10   
неполный день          0.61              0.35              0.49          0.10   
подработки             0.55              0.22              0.50          0.13   

                                                                 \
              alloc_работа_3 alloc_волонтёрство_3 alloc_курсы_3   
fmt_3                                                             
нет работы              0.20                 0.10          0.15   
полный день             0.40                 0.07          0.10   
неполный день           0.30                 0.05          0.10   
подработки              0.22                 0.09          0.10   

                                   n  
              alloc_остальное_3       
fmt_3                                 
нет работы                 0.25  678  
полный день                0.16   49  
неполный день              0.15  254  
подработки                 0.20  229

доля нулей                               \
                alloc_учёба_3 alloc_клубы_3 alloc_работа_3   
motive_raw_3                                                 
оплата обучения          0.17          0.67           0.25   
содержать себя           0.02          0.71           0.11   
доп. доход               0.05          0.70           0.16   
опыт                     0.05          0.63           0.16   
интерес                  0.05          0.71           0.12   
другое                   0.00          0.62           0.12   

                                                                      \
                alloc_волонтёрство_3 alloc_курсы_3 alloc_остальное_3   
motive_raw_3                                                           
оплата обучения                 0.92          0.58              0.75   
содержать себя                  0.88          0.62              0.29   
доп. доход                      0.85          0.60              0.25   
опыт                            0.84          0.57              0.43   
интерес                         0.86          0.52              0.27   
другое                          0.88          0.50              0.56   

                медиана среди > 0                               \
                    alloc_учёба_3 alloc_клубы_3 alloc_работа_3   
motive_raw_3                                                     
оплата обучения              0.60          0.07           0.40   
содержать себя               0.50          0.10           0.30   
доп. доход                   0.50          0.15           0.25   
опыт                         0.48          0.10           0.30   
интерес                      0.44          0.12           0.28   
другое                       0.50          0.18           0.26   

                                                                        n  
                alloc_волонтёрство_3 alloc_курсы_3 alloc_остальное_3       
motive_raw_3                                                               
оплата обучения                 0.31          0.20              0.20   12  
содержать себя                  0.05          0.10              0.15  178  
доп. доход                      0.03          0.10              0.20  183  
опыт                            0.08          0.13              0.20   87  
интерес                         0.11          0.17              0.16   56  
другое                          0.13          0.15              0.20   16

**Вывод.** В среднем половину времени студенты планируют на учёбу (медиана 50%). Нулевую долю на оплачиваемую работу указали 44%: среди неработающих — 67%, среди работающих полный день — 14%. Работающие полный день планируют на работу больше всего (медиана 40% среди указавших > 0). Студенческие клубы, волонтёрство и курсы большинство не планирует (нули у 63–89%). Различия по мотивам небольшие; группа «оплата обучения» мала (12 человек).

### Г6.2–Г6.3. Планы третьей волны и работа в четвёртой

Логистическая регрессия среди не работавших в 3-й волне и ответивших о работе в обеих волнах (набор `nw3`).

In [36]:
nw_rows = lambda f: f['start_work'].notna()
rows = [test('Г6.2', 'обе волны', 'logit', 'start_work', ['intent_3'], SOCIAL, nw_rows, 'nw3', +1),
        test('Г6.3', 'обе волны', 'logit', 'start_work', ['plan_work_pos_3'], SOCIAL, nw_rows, 'nw3', +1)]
ALL_ROWS += rows
display(show(report(rows)))
nw = d[d['start_work'].notna()]
display(pd.DataFrame({'намерение искать работу': shares_by(nw, 'start_work', 'intent_3'),
                      'доля времени на работу > 0': shares_by(nw, 'start_work', 'plan_work_pos_3')})
        .rename_axis('значение предиктора (0/1) → доля начавших работать'))
positive = nw[nw['plan_work_pos_3'].eq(1)]
print('Величина положительной доли (дополнительно): медиана у начавших работать —',
      median_iqr(positive.loc[positive['start_work'].eq(1), 'alloc_работа_3']),
      '; у не начавших —', median_iqr(positive.loc[positive['start_work'].eq(0), 'alloc_работа_3']))

,,модель,ожидаемый знак,A: n,эффект,A,B,C,D,p Холм (A),B: n,C: n,D: n,в ожидаемую сторону,вывод устойчив,поправки сокращены
гипотеза,проверка,,,,,,,,,,,,,,,
Г6.2,обе волны,логит,+,338,OR,"2.81 [1.62; 4.88], p = 0.000","2.99 [1.56; 5.75], p = 0.001","2.43 [1.36; 4.35], p = 0.003","2.79 [1.61; 4.85], p = 0.000",0.000,236,279,337,True,True,False
Г6.3,обе волны,логит,+,338,OR,"1.98 [1.15; 3.40], p = 0.013","2.12 [1.12; 4.02], p = 0.020","1.90 [1.05; 3.43], p = 0.033","2.01 [1.17; 3.45], p = 0.011",0.013,236,279,337,True,True,False


,намерение искать работу,доля времени на работу > 0
значение предиктора (0/1) → доля начавших работать,,
0.0,20% [15%; 26%] (40/200),23% [18%; 29%] (56/241)
1.0,38% [30%; 46%] (54/142),38% [29%; 47%] (38/101)


Величина положительной доли (дополнительно): медиана у начавших работать — 0.2 (0.1–0.4), n=38 ; у не начавших — 0.2 (0.1–0.3), n=63


**Вывод.**
- **Г6.2 подтверждается.** Среди не работавших в 3-й волне намеревавшиеся искать работу к 4-й волне работают вдвое чаще: 38% против 20%, OR = 2,81 [1,62; 4,88].
- **Г6.3 подтверждается.** Планировавшие уделять время работе работают чаще указавших ноль: 38% против 23%, OR = 1,98 [1,15; 3,40].

Оба эффекта устойчивы во всех вариантах. Величина положительной доли у начавших и не начавших работать одинакова (медиана 20%): важно само наличие плана, а не его размер.

### З6.4. Планы и последующие оценки и удовлетворённость

Порядковые модели 4-й волны: доля времени на работу (на 10 п. п.) с поправкой на исходный показатель, исходную занятость и общий набор. Устойчивость — без профилей A/B.

In [37]:
panel_rows = lambda f: f['panel']
no_profiles = lambda f: f['panel'] & ~f['tech_4w']
rows = []
for y, base_y, label in [('mark_4', 'mark_3', 'оценки'), ('sat_4', 'sat_3', 'удовлетворённость')]:
    rows.append(test('З6.4', f'{label}: доля на работу', 'ordinal', y, ['alloc10_3', base_y, 'emp_3'], SOCIAL,
                     panel_rows, 'w3', +1))
    rows.append(test('З6.4', f'{label}: доля > 0', 'ordinal', y, ['plan_work_pos_3', base_y, 'emp_3'], SOCIAL,
                     panel_rows, 'w3', +1))
    rows.append(test('З6.4', f'устойч.: {label}, доля на работу, без профилей A/B', 'ordinal', y,
                     ['alloc10_3', base_y, 'emp_3'], SOCIAL, no_profiles, 'w3', +1))
show(report(rows)).drop(columns=['ожидаемый знак', 'в ожидаемую сторону', 'p Холм (A)'])

модель  \
гипотеза проверка                                                                   
З6.4     оценки: доля на работу                                        порядковая   
         оценки: доля > 0                                              порядковая   
         устойч.: оценки, доля на работу, без профилей A/B             порядковая   
         удовлетворённость: доля на работу                             порядковая   
         удовлетворённость: доля > 0                                   порядковая   
         устойч.: удовлетворённость, доля на работу, без профилей A/B  порядковая   

                                                                       A: n  \
гипотеза проверка                                                             
З6.4     оценки: доля на работу                                         626   
         оценки: доля > 0                                               626   
         устойч.: оценки, доля на работу, без профилей A/B              527   
         удовлетворённость: доля на работу                              611   
         удовлетворённость: доля > 0                                    611   
         устойч.: удовлетворённость, доля на работу, без профилей A/B   511   

                                                                      эффект  \
гипотеза проверка                                                              
З6.4     оценки: доля на работу                                           OR   
         оценки: доля > 0                                                 OR   
         устойч.: оценки, доля на работу, без профилей A/B                OR   
         удовлетворённость: доля на работу                                OR   
         удовлетворённость: доля > 0                                      OR   
         устойч.: удовлетворённость, доля на работу, без профилей A/B     OR   

                                                                                                  A  \
гипотеза проверка                                                                                     
З6.4     оценки: доля на работу                                        1.08 [0.98; 1.17], p = 0.105   
         оценки: доля > 0                                              1.09 [0.76; 1.56], p = 0.647   
         устойч.: оценки, доля на работу, без профилей A/B             1.11 [1.00; 1.22], p = 0.046   
         удовлетворённость: доля на работу                             1.01 [0.91; 1.11], p = 0.921   
         удовлетворённость: доля > 0                                   0.88 [0.59; 1.34], p = 0.558   
         устойч.: удовлетворённость, доля на работу, без профилей A/B  1.02 [0.92; 1.15], p = 0.671   

                                                                                                  B  \
гипотеза проверка                                                                                     
З6.4     оценки: доля на работу                                        1.07 [0.96; 1.19], p = 0.201   
         оценки: доля > 0                                              0.97 [0.63; 1.49], p = 0.884   
         устойч.: оценки, доля на работу, без профилей A/B             1.12 [1.00; 1.26], p = 0.050   
         удовлетворённость: доля на работу                             0.95 [0.85; 1.07], p = 0.437   
         удовлетворённость: доля > 0                                   0.72 [0.44; 1.17], p = 0.183   
         устойч.: удовлетворённость, доля на работу, без профилей A/B  0.97 [0.85; 1.10], p = 0.640   

                                                                                                  C  \
гипотеза проверка                                                                                     
З6.4     оценки: доля на работу                                        1.06 [0.97; 1.17], p = 0.209   
         оценки: доля > 0                                              0.96 [0.65; 1.44], p = 0.855   
         устойч.: оценки, доля на работу, без профилей A/B

**Вывод (разведочно).** Планируемая доля времени на работу в 3-й волне не связана с оценками и удовлетворённостью в 4-й волне при учёте исходных показателей: OR = 1,08 (p = 0,11) и 1,01 на 10 процентных пунктов. Без профилей A/B для оценок OR = 1,11, p = 0,046 — на границе значимости, не устойчиво.

## 14. ИВ7. Изменения между волнами

### З7.1. Переходы между состояниями занятости

In [38]:
display(pd.Series({t: share(d['trans4'], t) for t in ['остался неработающим', 'начал работать', 'прекратил работу',
                                                       'работал в обеих']}, name=f"n = {d['trans4'].notna().sum()}").to_frame())
display(pd.crosstab(d['fmt_3'], d['fmt_4'], margins=True).rename_axis(index='3-я волна', columns='4-я волна'))
both = d[d['trans4'].eq('работал в обеих')]
print('Изменение часов у работавших в обеих волнах: медиана (IQR) —', median_iqr(both['d_hours']))

,n = 566
остался неработающим,44% [40%; 48%] (248/566)
начал работать,17% [14%; 20%] (94/566)
прекратил работу,13% [10%; 16%] (71/566)
работал в обеих,27% [24%; 31%] (153/566)


4-я волна,нет работы,полный день,неполный день,подработки,All
3-я волна,,,,,
нет работы,248,14,38,42,342
полный день,5,8,6,1,20
неполный день,27,13,49,18,107
подработки,39,6,19,33,97
All,319,41,112,94,566


Изменение часов у работавших в обеих волнах: медиана (IQR) — 0.0 (-8.5–10.0), n=127


**Вывод.** Из 566 человек с ответом о работе в обеих волнах: 44% не работали ни в одной, 17% начали работать, 13% прекратили, 27% работали в обеих. Начавшие работать чаще выходили на подработки (42) и неполный день (38), чем на полный день (14). У работавших в обеих волнах медиана изменения часов — 0 (межквартильный размах от −8,5 до +10).

### Г7.2–Г7.3. Начало работы, удовлетворённость и оценки

Порядковая регрессия показателя 4-й волны на начало работы с поправкой на тот же показатель в 3-й волне и общий набор. Среди не работавших в 3-й волне (набор `nw3`).

In [39]:
rows = [test('Г7.2', 'обе волны', 'ordinal', 'sat_4', ['start_work', 'sat_3'], SOCIAL, nw_rows, 'nw3', -1),
        test('Г7.3', 'обе волны', 'ordinal', 'mark_4', ['start_work', 'mark_3'], SOCIAL, nw_rows, 'nw3', -1)]
ALL_ROWS += rows
display(show(report(rows)))
starters = d[d['trans4'].eq('начал работать')]
pd.DataFrame({'n': starters['start_fmt'].value_counts(),
              'удовлетворённость 4-й волны': starters.groupby('start_fmt', observed=True)['sat_4'].apply(mean_ci),
              'оценки 4-й волны': starters.groupby('start_fmt', observed=True)['mark_4'].apply(mean_ci)})

,,модель,ожидаемый знак,A: n,эффект,A,B,C,D,p Холм (A),B: n,C: n,D: n,в ожидаемую сторону,вывод устойчив,OR по порогам (B),поправки сокращены
гипотеза,проверка,,,,,,,,,,,,,,,,
Г7.2,обе волны,порядковая,−,304,OR,"1.13 [0.66; 1.96], p = 0.652","1.33 [0.67; 2.63], p = 0.417","1.17 [0.64; 2.17], p = 0.607","1.13 [0.66; 1.96], p = 0.652",0.652,211,250,304,False,True,≥2: —; ≥3: 2.02; ≥4: 1.02,False
Г7.3,обе волны,порядковая,−,315,OR,"0.93 [0.56; 1.54], p = 0.787","1.02 [0.55; 1.90], p = 0.956","0.86 [0.49; 1.50], p = 0.597","0.93 [0.56; 1.54], p = 0.784",0.787,220,260,314,True,True,≥2: 0.65; ≥3: 0.93; ≥4: 1.13,False


,n,удовлетворённость 4-й волны,оценки 4-й волны
start_fmt,,,
нет работы,0,NaN,NaN
полный день,14,"3.14 [2.83; 3.45], n=14","2.46 [1.83; 3.10], n=13"
неполный день,38,"3.30 [3.12; 3.49], n=33","2.74 [2.46; 3.01], n=38"
подработки,42,"3.05 [2.87; 3.23], n=39","2.67 [2.32; 3.03], n=40"


**Вывод.**
- **Г7.2 не подтверждается.** Начавшие работать не стали менее удовлетворены обучением: OR = 1,13 [0,66; 1,96] при учёте исходной удовлетворённости.
- **Г7.3 не подтверждается.** Их оценки не снизились: OR = 0,93 [0,56; 1,54].

Оба вывода устойчивы. У 14 начавших работать полный день средняя категория оценок ниже (2,46 против 2,67–2,74 у других форматов), но группа слишком мала для вывода.

### Г7.4. Изменение рабочих часов и сна

Линейная регрессия изменения сна в будни на изменение часов (на 10 ч) с поправкой на исходные часы, исходный сон и общий набор. Среди работавших в обеих волнах (набор `workboth`).

In [40]:
rows = [test('Г7.4', 'обе волны', 'ols', 'd_sleep_wd', ['d_h10', 'h10_3', 'sleep_dur_wd_3'], SOCIAL,
             lambda f: f['trans4'].eq('работал в обеих'), 'workboth', -1)]
ALL_ROWS += rows
show(report(rows))

,,модель,ожидаемый знак,A: n,эффект,A,B,C,D,p Холм (A),B: n,C: n,D: n,в ожидаемую сторону,вывод устойчив,поправки сокращены
гипотеза,проверка,,,,,,,,,,,,,,,
Г7.4,обе волны,"МНК, HC3",−,152,b,"0.03 [-0.19; 0.26], p = 0.783","-0.01 [-0.32; 0.30], p = 0.926","-0.02 [-0.24; 0.20], p = 0.861","0.10 [-0.16; 0.37], p = 0.443",0.783,86,99,120,False,True,False


**Вывод. Г7.4 не подтверждается.** У работавших в обеих волнах изменение часов не связано с изменением сна в будни: b = 0,03 часа на 10 часов [−0,19; 0,26]. Выборка мала (152 человека в основном варианте, 86 в полных случаях), а медиана изменения часов — 0, поэтому изменчивость, на которой можно увидеть эффект, невелика.

### Г7.5. Опыт стажировки и выход на работу

Логистическая регрессия среди не работавших в 3-й волне; дополнительная модель — с намерением искать работу.

In [41]:
rows = [test('Г7.5', 'обе волны', 'logit', 'start_work', ['intern_3'], SOCIAL, nw_rows, 'nw3', +1),
        test('Г7.5', 'устойч.: + намерение искать работу', 'logit', 'start_work', ['intern_3', 'intent_3'], SOCIAL,
             nw_rows, 'nw3', +1)]
ALL_ROWS += rows
display(show(report(rows)))
shares_by(d[d['start_work'].notna()], 'start_work', 'intern_3').rename('доля начавших работать').to_frame()

модель ожидаемый знак  A: n  \
гипотеза проверка                                                         
Г7.5     обе волны                           логит              +   338   
         устойч.: + намерение искать работу  логит              +   338   

                                            эффект  \
гипотеза проверка                                    
Г7.5     обе волны                              OR   
         устойч.: + намерение искать работу     OR   

                                                                        A  \
гипотеза проверка                                                           
Г7.5     обе волны                           0.93 [0.41; 2.12], p = 0.863   
         устойч.: + намерение искать работу  0.87 [0.37; 2.04], p = 0.744   

                                                                        B  \
гипотеза проверка                                                           
Г7.5     обе волны                           1.32 [0.49; 3.58], p = 0.586   
         устойч.: + намерение искать работу  1.17 [0.42; 3.32], p = 0.762   

                                                                        C  \
гипотеза проверка                                                           
Г7.5     обе волны                           1.00 [0.39; 2.58], p = 1.000   
         устойч.: + намерение искать работу  0.90 [0.34; 2.40], p = 0.836   

                                                                        D  \
гипотеза проверка                                                           
Г7.5     обе волны                           0.92 [0.40; 2.10], p = 0.842   
         устойч.: + намерение искать работу  0.86 [0.37; 2.03], p = 0.731   

                                             p Холм (A)  B: n  C: n  D: n  \
гипотеза проверка                                                           
Г7.5     обе волны                                0.863   236   279   337   
         устойч.: + намерение искать работу       0.744   236   279   337   

                                             в ожидаемую сторону  \
гипотеза проверка                                                  
Г7.5     обе волны                                         False   
         устойч.: + намерение искать работу                False   

                                             вывод устойчив  \
гипотеза проверка                                             
Г7.5     обе волны                                     True   
         устойч.: + намерение искать работу            True   

                                             поправки сокращены  
гипотеза проверка                                                
Г7.5     обе волны                                        False  
         устойч.: + намерение искать работу               False

,доля начавших работать
intern_3,
0.0,27% [23%; 33%] (84/306)
1.0,28% [16%; 44%] (10/36)


**Вывод. Г7.5 не подтверждается.** Опыт стажировки не связан с выходом на работу к 4-й волне: начали работать 28% стажировавшихся и 27% остальных (OR = 0,93 [0,41; 2,12]). С учётом намерения искать работу — то же самое. Стажировавшихся среди не работавших в 3-й волне всего 36, поэтому интервал очень широкий.

### З7.6. Изменения при разных переходах занятости

Изменения курения, алкоголя и спорта — только по содержательным ответам в обеих волнах.

In [42]:
trans = d[d['trans4'].notna()].assign(d_sat=lambda f: f['sat_4'] - f['sat_3'], d_mark=lambda f: f['mark_4'] - f['mark_3'],
                                      same_edu=lambda f: (f['edu3_3'].astype(object) == f['edu3_4'].astype(object))
                                      .astype(float).where(f['edu3_3'].notna() & f['edu3_4'].notna()))
ORDER7 = ['остался неработающим', 'начал работать', 'прекратил работу', 'работал в обеих']
changes = pd.DataFrame({'Δ удовлетворённости': trans.groupby('trans4')['d_sat'].apply(mean_ci),
                        'Δ оценок': trans.groupby('trans4')['d_mark'].apply(mean_ci),
                        'Δ сна в будни, ч': trans.groupby('trans4')['d_sleep_wd'].apply(mean_ci),
                        'тот же повторный выбор': shares_by(trans, 'same_edu', 'trans4'),
                        'впервые думал об отчислении': shares_by(trans, 'new_drop_4', 'trans4')}).reindex(ORDER7)
changes.loc['p'] = [stats.kruskal(*[g.dropna() for _, g in trans.groupby('trans4')[c]]).pvalue for c in ['d_sat', 'd_mark', 'd_sleep_wd']] + \
                   [assoc(trans['trans4'], trans[c])['p'] for c in ['same_edu', 'new_drop_4']]
display(changes.T)
for v in ['smoke_change', 'alco_change', 'pe_change']:
    table = pd.crosstab(trans['trans4'], trans[v], margins=True).reindex(ORDER7 + ['All'])
    display(table.rename_axis(f"{v}: p = {assoc(trans['trans4'], trans[v])['p']:.3f}"))

trans4,остался неработающим,начал работать,прекратил работу,работал в обеих,p
Δ удовлетворённости,"-0.04 [-0.13; 0.06], n=225","-0.05 [-0.21; 0.11], n=82","0.00 [-0.25; 0.25], n=60","-0.06 [-0.19; 0.07], n=141",0.544706
Δ оценок,"0.19 [0.08; 0.29], n=230","0.18 [-0.02; 0.38], n=89","0.06 [-0.14; 0.27], n=65","0.11 [-0.04; 0.25], n=142",0.559879
"Δ сна в будни, ч","0.24 [0.01; 0.48], n=247","0.34 [-0.02; 0.69], n=92","-0.04 [-0.61; 0.52], n=69","0.55 [0.26; 0.83], n=152",0.247642
тот же повторный выбор,60% [53%; 65%] (147/247),47% [37%; 57%] (44/94),55% [43%; 66%] (38/69),59% [51%; 66%] (90/153),0.179502
впервые думал об отчислении,17% [12%; 23%] (27/161),12% [6%; 23%] (7/59),15% [7%; 28%] (6/41),33% [23%; 44%] (25/76),0.006266


smoke_change,0→0,0→1,1→0,1→1,All
smoke_change: p = 0.000,,,,,
остался неработающим,183,4,9,25,221
начал работать,53,12,3,14,82
прекратил работу,43,1,5,8,57
работал в обеих,89,8,12,22,131
All,368,25,29,69,491


alco_change,0→0,0→1,1→0,1→1,All
alco_change: p = 0.044,,,,,
остался неработающим,98,21,13,85,217
начал работать,23,9,10,34,76
прекратил работу,20,3,9,22,54
работал в обеих,44,15,7,62,128
All,185,48,39,203,475


pe_change,0→0,0→1,1→0,1→1,All
pe_change: p = 0.047,,,,,
остался неработающим,59,19,69,100,247
начал работать,13,11,22,47,93
прекратил работу,12,5,15,39,71
работал в обеих,20,11,35,87,153
All,104,46,141,273,564


**Вывод (разведочно).**
- Изменения удовлетворённости и оценок между волнами не различаются по переходам занятости (p = 0,54 и 0,56): во всех группах оценки немного выросли, удовлетворённость не изменилась.
- Впервые задумались об отчислении 33% работавших в обеих волнах против 12–17% в других группах (p = 0,006).
- Курить начали 15% начавших работать (12 из 82) против 2% оставшихся неработающими (p < 0,001).
- Изменения алкоголя и спорта различаются слабо (p = 0,044 и 0,047).
- Ответ о повторном выборе образования совпал в двух волнах у 47–60% (p = 0,18).

Это описательные сравнения без поправок; причинно их интерпретировать нельзя.

## 15. Сводка решений по гипотезам

Решение по варианту A с поправкой Холма: «подтверждается» — эффект в ожидаемую сторону и значим во всех основных проверках гипотезы; «частично» — в части проверок (например, в одной волне); «не подтверждается» — ни в одной. «Устойчиво в B–D» — тот же вывод в полных случаях, без материального положения и без сомнительных значений.

In [43]:
summary = report(ALL_ROWS)
verdict(summary)

,проверок,подтвердилось,решение,устойчиво в B–D
гипотеза,,,,
Г1.2,2,0,не подтверждается,True
Г1.3,2,0,не подтверждается,True
Г1.4,2,0,не подтверждается,True
Г2.1,2,0,не подтверждается,True
Г2.2,2,0,не подтверждается,True
Г2.3,2,0,не подтверждается,False
Г2.4,2,0,не подтверждается,False
Г3.1,1,1,подтверждается,True
Г3.2,1,0,не подтверждается,True


**Итог.** Из 25 гипотез Г подтверждаются 8, частично — 1, не подтверждаются — 16.

| Решение | Гипотезы |
|---|---|
| Подтверждаются | Г3.1, Г3.3, Г3.5 (польза и трудности работы); Г4.2, Г4.3, Г4.4 (стажировки); Г6.2, Г6.3 (планы и последующая работа) |
| Частично | Г5.1 (часы и сон — только 3-я волна) |
| Не подтверждаются | Г1.2–Г1.4, Г2.1–Г2.4, Г3.2, Г3.4, Г5.2–Г5.4, Г7.2–Г7.5 |

**Неустойчивые выводы** (в вариантах B–D решение другое):
- **Г4.4 и Г5.1** подтверждаются только в основном варианте.
- **Г2.3 и Г2.4** не подтверждаются в основном варианте, но подтверждаются в полных случаях (Г2.3) или без сомнительных значений часов (Г2.4). В обоих случаях направление совпадает с гипотезой, а p близко к порогу. Их стоит описывать как «направление ожидаемое, доказательства недостаточно сильные».

**Общая картина.** Само наличие и количество работы мало связано с оценками, удовлетворённостью и здоровьем. Значение имеет качество занятости: соответствие специальности (для работы и стажировки) и мотив. Работающие ради содержания себя испытывают больше трудностей совмещения.